### Setup

In [1]:
from pathlib import Path
import sys
import torch
import torch.nn as nn

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            (candidate / "main.py").exists()
            and
            (candidate / "utils").exists()
            and
            (candidate / "thesis").exists()
        ):
            return candidate

    raise RuntimeError(
        "PyCIL project root not found."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


from thesis.backbones.resnet_attention import (
    resnet18_remote,
    resnet34_remote,
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "CUDA:",
    torch.cuda.is_available(),
)

ImportError: cannot import name 'get_ssre_backbone_variant' from 'thesis.backbones.resnet_ssre' (C:\Skripsi\Project\PyCIL\thesis\backbones\resnet_ssre.py)

### Ensure no CIFAR override


In [ ]:
import inspect

from models.fetril import FeTrIL


source = inspect.getsource(
    FeTrIL.incremental_train
)

assert "RandomCrop(32" not in source
assert "_train_trsf" not in source

print(
    "FeTrIL transform override removed: PASSED"
)

### Ensure backbone work on FeTriL

In [ ]:
import torch

from thesis.backbones.variants import (
    get_all_backbone_variants,
)


device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


base_args = {
    "dataset": "aid",

    "memory_size": 0,
    "fixed_memory": False,

    "device": [device],

    "batch_size": 2,
    "num_workers": 0,

    "seed": 1993,

    "fetril_classifier": "svm",

    "svm_c": 1.0,
    "svm_tol": 1e-4,

    "init_epochs": 1,
    "init_lr": 0.1,
    "init_weight_decay": 5e-4,

    "epochs": 1,
    "lr": 0.05,
    "weight_decay": 5e-4,
}


for name, backbone_args in (
    get_all_backbone_variants()
    .items()
):

    args = {
        **base_args,
        **backbone_args,
    }

    learner = FeTrIL(
        args
    )

    learner._network.update_fc(
        14
    )

    learner._network.eval()

    x = torch.randn(
        1,
        3,
        256,
        256,
    )

    with torch.no_grad():
        output = (
            learner._network(x)
        )

    assert (
        output["features"].shape
        == (1, 512)
    )

    assert (
        output["logits"].shape
        == (1, 14)
    )

    print(
        name,
        ": PASSED"
    )

### Validate freeze FeTriL

In [2]:
from thesis.backbones.variants import (
    get_backbone_variant,
)


for variant in [
    "r18_eca",
    "r18_cbam",
    "r18_procbam",
]:

    args = {
        **base_args,
        **get_backbone_variant(
            variant
        ),
    }

    learner = FeTrIL(args)

    learner._network.update_fc(
        14
    )

    learner._freeze_backbone()

    convnet_trainable = sum(
        p.requires_grad
        for p
        in learner
        ._network
        .convnet
        .parameters()
    )

    fc_trainable = sum(
        p.requires_grad
        for p
        in learner
        ._network
        .fc
        .parameters()
    )

    print(
        variant,
        "convnet:",
        convnet_trainable,
        "fc:",
        fc_trainable,
    )

    assert convnet_trainable == 0
    assert fc_trainable > 0


print(
    "FeTrIL frozen backbone: PASSED"
)

NameError: name 'base_args' is not defined

### Unit test SVM without training image

In [3]:
import numpy as np

from models.fetril import (
    FeatureDataset,
)


rng = np.random.RandomState(
    1993
)

num_classes = 14
feature_dim = 512


train_features = []
train_labels = []

test_features = []
test_labels = []


for c in range(
    num_classes
):

    center = np.zeros(
        feature_dim,
        dtype=np.float32,
    )

    center[c] = 10.0

    for _ in range(5):

        train_features.append(
            center
            + rng.normal(
                0,
                0.01,
                feature_dim,
            )
        )

        train_labels.append(c)

    for _ in range(2):

        test_features.append(
            center
            + rng.normal(
                0,
                0.01,
                feature_dim,
            )
        )

        test_labels.append(c)


train_set = FeatureDataset(
    np.asarray(
        train_features,
        dtype=np.float32,
    ),
    np.asarray(
        train_labels,
        dtype=np.int64,
    ),
)

test_set = FeatureDataset(
    np.asarray(
        test_features,
        dtype=np.float32,
    ),
    np.asarray(
        test_labels,
        dtype=np.int64,
    ),
)


args = {
    **base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}

learner = FeTrIL(args)

learner._feature_trainset = (
    train_set
)

learner._feature_testset = (
    test_set
)

learner._train_svm(
    train_set,
    test_set,
)

pred, true = learner._eval_svm()

top1 = (
    pred[:, 0] == true
).mean()


print(
    "Synthetic SVM top1:",
    top1,
)

assert top1 > 0.99

print(
    "FeTrIL SVM path: PASSED"
)

NameError: name 'base_args' is not defined

### Validasi reproducibility seed

In [ ]:
import random
import numpy as np
import torch

from trainer import _set_random


_set_random(1993)

a_python = random.random()
a_numpy = np.random.rand()
a_torch = torch.rand(1)


_set_random(1993)

b_python = random.random()
b_numpy = np.random.rand()
b_torch = torch.rand(1)


assert a_python == b_python
assert a_numpy == b_numpy
assert torch.equal(
    a_torch,
    b_torch,
)


print(
    "Training seed reproducibility: PASSED"
)

## Full Fetril smoke test

### Configure smoke test

In [4]:
from pathlib import Path

import gc
import time

import numpy as np
import torch

from torch.utils.data import (
    WeightedRandomSampler,
)

from models.fetril import FeTrIL
from trainer import _set_random
from utils.data_manager import DataManager

from thesis.backbones.variants import (
    get_backbone_variant,
)

In [5]:
SMOKE_SEED = 1993

smoke_args = {
    # =========================
    # Dataset
    # =========================
    "dataset": "aid",

    "dataset_root": str(
        PROJECT_ROOT
        / "datasets/raw/AID"
    ),

    "split_root": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993"
    ),

    "task_config_path": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993/T5_tasks.json"
    ),

    "shuffle": False,

    # =========================
    # Method
    # =========================
    "model_name": "fetril",

    "memory_size": 0,
    "fixed_memory": False,

    "fetril_classifier": "svm",

    "svm_c": 1.0,
    "svm_tol": 1e-4,

    # =========================
    # Smoke training
    # =========================
    "init_epochs": 1,

    "init_lr": 0.1,

    "init_weight_decay": 5e-4,

    # Tidak dipakai canonical SVM
    # pada incremental tasks,
    # tetapi tetap disediakan.
    "epochs": 1,

    "lr": 0.05,

    "weight_decay": 5e-4,

    # AID 600x600.
    # Ini SMOKE batch size,
    # belum final experimental batch.
    "batch_size": 4,

    # Notebook Windows:
    # 0 paling aman.
    "num_workers": 0,

    # =========================
    # Reproducibility
    # =========================
    "seed": SMOKE_SEED,

    # BaseLearner membutuhkan
    # list torch.device.
    "device": [
        torch.device(
            "cuda:0"
            if torch.cuda.is_available()
            else "cpu"
        )
    ],

    # =========================
    # Backbone
    # =========================
    **get_backbone_variant(
        "r18_none"
    ),
}

### Create Data Manager

In [6]:
_set_random(
    SMOKE_SEED
)


data_manager = DataManager(
    dataset_name=(
        smoke_args["dataset"]
    ),

    shuffle=(
        smoke_args["shuffle"]
    ),

    seed=(
        SMOKE_SEED
    ),

    init_cls=14,
    increment=4,

    dataset_root=(
        smoke_args[
            "dataset_root"
        ]
    ),

    split_root=(
        smoke_args[
            "split_root"
        ]
    ),

    task_config_path=(
        smoke_args[
            "task_config_path"
        ]
    ),
)


print(
    "Tasks:",
    data_manager.nb_tasks
)

print(
    "Task sizes:",
    [
        data_manager
        .get_task_size(i)

        for i in range(
            data_manager.nb_tasks
        )
    ]
)


assert (
    data_manager.nb_tasks
    == 5
)

assert [
    data_manager
    .get_task_size(i)

    for i in range(
        data_manager.nb_tasks
    )
] == [
    14,
    4,
    4,
    4,
    4,
]


print(
    "Smoke DataManager: PASSED"
)

Tasks: 5
Task sizes: [14, 4, 4, 4, 4]
Smoke DataManager: PASSED


### Initialize Fetril

In [7]:
learner = FeTrIL(
    smoke_args
)


print(
    "Initial known classes:",
    learner._known_classes
)

print(
    "Current task:",
    learner._cur_task
)

print(
    "Feature dim:",
    learner.feature_dim
)


assert (
    learner._known_classes
    == 0
)

assert (
    learner._cur_task
    == -1
)

assert (
    learner.feature_dim
    == 512
)


print(
    "FeTrIL initialization: PASSED"
)

Initial known classes: 0
Current task: -1
Feature dim: 512
FeTrIL initialization: PASSED


### Reset VRAM Statistics ?

In [ ]:
if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


gc.collect()

### Run Task 0

In [ ]:
task0_start = time.perf_counter()
learner.incremental_train(data_manager)
task0_seconds = (time.perf_counter() - task0_start)

print(
    "Task 0 runtime:",
    round(
        task0_seconds / 60,
        2,
    ),
    "minutes",
)

### Validate Task 0

In [ ]:
print(
    "Current task:",
    learner._cur_task
)

print(
    "Total classes:",
    learner._total_classes
)

print(
    "Means:",
    len(
        learner._means
    )
)

print(
    "SVM classes:",
    learner
    ._svm_classifier
    .classes_
)

print(
    "Feature train shape:",
    learner
    ._feature_trainset
    .features
    .shape
)

print(
    "Feature test shape:",
    learner
    ._feature_testset
    .features
    .shape
)

print(
    "Train sampler:",
    type(
        learner
        .train_loader
        .sampler
    ).__name__,
)

print(
    "Exemplars:",
    learner.exemplar_size
)

In [ ]:
assert learner._cur_task == 0

assert (
    learner._total_classes
    == 14
)

assert (
    len(
        learner._means
    )
    == 14
)

assert np.array_equal(
    learner
    ._svm_classifier
    .classes_,

    np.arange(14),
)

assert (
    learner
    ._feature_trainset
    .features
    .shape[1]
    == 512
)

assert (
    learner
    ._feature_testset
    .features
    .shape[1]
    == 512
)

assert isinstance(
    learner
    .train_loader
    .sampler,

    WeightedRandomSampler,
)

assert (
    learner.exemplar_size
    == 0
)

print(
    "FeTrIL Task 0 structure: PASSED"
)

### Evaluate SVM Task 0

In [ ]:
task0_accy, _ = (
    learner.eval_task()
)


print(
    "Task 0 SVM grouped:"
)

print(
    task0_accy[
        "grouped"
    ]
)

print(
    "Task 0 top-1:",
    task0_accy[
        "top1"
    ]
)

print(
    "Task 0 top-5:",
    task0_accy[
        "top5"
    ]
)


assert (
    0.0
    <= task0_accy[
        "top1"
    ]
    <= 100.0
)

assert np.isfinite(
    task0_accy[
        "top1"
    ]
)


print(
    "Task 0 SVM evaluation: PASSED"
)

In [ ]:
learner.after_task()


assert (
    learner._known_classes
    == 14
)

print(
    "Task 0 after_task: PASSED"
)

### Check peak of VRAM initial task

In [ ]:
if torch.cuda.is_available():

    peak_allocated = (
        torch.cuda
        .max_memory_allocated()
        / 1024**3
    )

    peak_reserved = (
        torch.cuda
        .max_memory_reserved()
        / 1024**3
    )

    print(
        "Peak allocated:",
        round(
            peak_allocated,
            2,
        ),
        "GB",
    )

    print(
        "Peak reserved:",
        round(
            peak_reserved,
            2,
        ),
        "GB",
    )

### Run Task 1 seperately

In [ ]:
task1_start = time.perf_counter()


learner.incremental_train(
    data_manager
)


task1_seconds = (
    time.perf_counter()
    - task1_start
)


print(
    "Task 1 runtime:",
    round(
        task1_seconds / 60,
        2,
    ),
    "minutes",
)

### Validate task 1

In [ ]:
print(
    "Current task:",
    learner._cur_task
)

print(
    "Known before after_task:",
    learner._known_classes
)

print(
    "Total classes:",
    learner._total_classes
)

print(
    "Means:",
    len(
        learner._means
    )
)

print(
    "Relations:",
    learner._relations.shape
)

print(
    "Train loader:",
    learner.train_loader
)

print(
    "SVM classes:",
    learner
    ._svm_classifier
    .classes_
)

In [ ]:
assert learner._cur_task == 1

assert (
    learner._known_classes
    == 14
)

assert (
    learner._total_classes
    == 18
)

assert (
    len(
        learner._means
    )
    == 18
)

assert (
    learner._relations.shape
    == (14,)
)

assert (
    learner.train_loader
    is None
)

assert np.array_equal(
    learner
    ._svm_classifier
    .classes_,

    np.arange(18),
)

In [ ]:
trainable_backbone = sum(
    parameter.numel()

    for parameter
    in learner
    ._network
    .convnet
    .parameters()

    if parameter.requires_grad
)


print(
    "Trainable backbone params:",
    trainable_backbone
)


assert (
    trainable_backbone
    == 0
)

print(
    "FeTrIL Task 1 incremental "
    "behavior: PASSED"
)

### Check pseudo feature set

In [ ]:
feature_labels = (
    learner
    ._feature_trainset
    .labels
    .numpy()
)


unique_labels = np.unique(
    feature_labels
)


print(
    "Feature labels:",
    unique_labels
)

print(
    "Number of feature samples:",
    len(
        feature_labels
    )
)


assert np.array_equal(
    unique_labels,
    np.arange(18),
)


print(
    "Old + new feature classes: "
    "PASSED"
)

### Evaluate task 1

In [ ]:
task1_accy, _ = (
    learner.eval_task()
)


print(
    "Task 1 top-1:",
    task1_accy[
        "top1"
    ]
)

print(
    "Task 1 grouped:",
    task1_accy[
        "grouped"
    ]
)


assert np.isfinite(
    task1_accy[
        "top1"
    ]
)

learner.after_task()


assert (
    learner._known_classes
    == 18
)


print(
    "Task 1 evaluation: PASSED"
)

### Run the rest Task 2-4

In [ ]:
top1_curve = [
    task0_accy["top1"],
    task1_accy["top1"],
]

task_runtime_minutes = [
    task0_seconds / 60,
    task1_seconds / 60,
]

In [ ]:
for expected_task in range(
    2,
    data_manager.nb_tasks,
):

    start = time.perf_counter()

    learner.incremental_train(
        data_manager
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    accy, _ = (
        learner.eval_task()
    )

    expected_total = (
        data_manager
        .get_accumulate_tasksize(
            expected_task
        )
    )

    print()
    print(
        "=" * 60
    )

    print(
        "Task:",
        expected_task
    )

    print(
        "Total classes:",
        learner._total_classes
    )

    print(
        "Means:",
        len(
            learner._means
        )
    )

    print(
        "SVM classes:",
        len(
            learner
            ._svm_classifier
            .classes_
        )
    )

    print(
        "Top-1:",
        accy["top1"]
    )

    print(
        "Runtime:",
        round(
            elapsed / 60,
            2,
        ),
        "minutes",
    )


    assert (
        learner._cur_task
        == expected_task
    )

    assert (
        learner._total_classes
        == expected_total
    )

    assert (
        len(
            learner._means
        )
        == expected_total
    )

    assert (
        len(
            learner
            ._svm_classifier
            .classes_
        )
        == expected_total
    )

    assert (
        learner.train_loader
        is None
    )

    assert (
        learner.exemplar_size
        == 0
    )

    assert np.isfinite(
        accy["top1"]
    )


    top1_curve.append(
        accy["top1"]
    )

    task_runtime_minutes.append(
        elapsed / 60
    )


    learner.after_task()

### Final smoke validation

In [ ]:
print(
    "Top-1 curve:",
    top1_curve
)

print(
    "Task runtime minutes:",
    [
        round(x, 2)
        for x
        in task_runtime_minutes
    ]
)


average_incremental_accuracy = (
    float(
        np.mean(
            top1_curve
        )
    )
)

final_accuracy = (
    float(
        top1_curve[-1]
    )
)


print(
    "Smoke AIA:",
    round(
        average_incremental_accuracy,
        2,
    )
)

print(
    "Smoke final accuracy:",
    round(
        final_accuracy,
        2,
    )
)


assert (
    len(
        top1_curve
    )
    == 5
)

assert (
    learner._known_classes
    == 30
)

assert (
    learner._total_classes
    == 30
)

assert (
    len(
        learner._means
    )
    == 30
)

assert (
    len(
        learner._svm_accs
    )
    == 5
)

assert (
    learner.exemplar_size
    == 0
)


print(
    "FULL FeTrIL T5 smoke test: "
    "PASSED"
)

## Benchmark with 8 backbone and AID dataset on FeTriL

### Setup

In [11]:
import gc
import time

import numpy as np
import pandas as pd
import torch

from torch import optim
from torch.nn import functional as F
from torch.utils.data import (
    WeightedRandomSampler,
)

from models.fetril import FeTrIL

from thesis.backbones.variants import (
    get_all_backbone_variants,
)

from thesis.data.loaders import (
    build_train_loader,
)

from trainer import _set_random

### Take AID Task 0

In [12]:
task0_classes = (
    data_manager.get_task_size(0)
)

task0_train_dataset = (
    data_manager.get_dataset(
        np.arange(
            0,
            task0_classes,
        ),
        source="train",
        mode="train",
    )
)


print(
    "Task 0 classes:",
    task0_classes
)

print(
    "Task 0 samples:",
    len(
        task0_train_dataset
    )
)


assert task0_classes == 14

assert (
    len(
        task0_train_dataset
    )
    == 3656
)


print(
    "Benchmark dataset: PASSED"
)

Task 0 classes: 14
Task 0 samples: 3656
Benchmark dataset: PASSED


### Benchmark function



In [13]:
BENCH_BATCH_SIZE = 4
WARMUP_STEPS = 3
BENCH_STEPS = 30


def benchmark_fetril_variant(
    variant_name,
    backbone_config,
):

    _set_random(
        SMOKE_SEED
    )

    args = {
        **smoke_args,
        **backbone_config,

        "batch_size":
            BENCH_BATCH_SIZE,

        "num_workers":
            0,
    }


    learner = FeTrIL(
        args
    )

    network = (
        learner._network
    )

    network.update_fc(
        nb_classes=14
    )

    device = args[
        "device"
    ][0]

    network = network.to(
        device
    )

    network.train()


    loader = build_train_loader(
        dataset=(
            task0_train_dataset
        ),

        dataset_name="aid",

        batch_size=(
            BENCH_BATCH_SIZE
        ),

        num_workers=0,

        seed=(
            SMOKE_SEED
        ),
    )


    assert isinstance(
        loader.sampler,
        WeightedRandomSampler,
    )


    optimizer = optim.SGD(
        network.parameters(),

        lr=0.1,

        momentum=0.9,

        weight_decay=5e-4,
    )


    iterator = iter(loader)


    # =========================
    # Warm-up
    # =========================
    for _ in range(
        WARMUP_STEPS
    ):

        _, inputs, targets = next(
            iterator
        )

        targets = targets.long()

        inputs = inputs.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = network(
            inputs
        )["logits"]

        loss = F.cross_entropy(
            logits,
            targets,
        )

        loss.backward()

        optimizer.step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    # =========================
    # Timed section
    # =========================
    start = time.perf_counter()

    last_loss = None


    for _ in range(
        BENCH_STEPS
    ):

        _, inputs, targets = next(
            iterator
        )

        targets = targets.long()

        inputs = inputs.to(
            device,
            non_blocking=True,
        )

        targets = targets.to(
            device,
            non_blocking=True,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = network(
            inputs
        )["logits"]


        loss = F.cross_entropy(
            logits,
            targets,
        )


        loss.backward()

        optimizer.step()

        last_loss = float(
            loss.detach().cpu()
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    seconds_per_step = (
        elapsed
        / BENCH_STEPS
    )


    projected_epoch_seconds = (
        seconds_per_step
        * len(loader)
    )


    attention_grad_params = sum(
        1
        for name, parameter
        in network.named_parameters()

        if (
            ".attention." in name
            and
            parameter.grad is not None
        )
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    result = {
        "variant":
            variant_name,

        "batch_size":
            BENCH_BATCH_SIZE,

        "steps_per_epoch":
            len(loader),

        "sec_per_step":
            seconds_per_step,

        "projected_epoch_min":
            projected_epoch_seconds
            / 60,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "last_loss":
            last_loss,
    }


    assert np.isfinite(
        last_loss
    )


    attention_type = (
        backbone_config[
            "attention_type"
        ]
    )


    if attention_type == "none":

        assert (
            attention_grad_params
            == 0
        )

    else:

        assert (
            attention_grad_params
            > 0
        )


    del learner
    del network
    del optimizer
    del loader
    del iterator
    del inputs
    del targets
    del logits
    del loss


    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return result

### Run all 8 variants

In [ ]:
benchmark_results = []


variants = (
    get_all_backbone_variants()
)


for variant_name, config in (
    variants.items()
):

    print()
    print(
        "=" * 60
    )

    print(
        "Benchmark:",
        variant_name
    )


    result = (
        benchmark_fetril_variant(
            variant_name,
            config,
        )
    )


    benchmark_results.append(
        result
    )


    print(
        "sec/step:",
        round(
            result[
                "sec_per_step"
            ],
            4,
        )
    )

    print(
        "projected epoch:",
        round(
            result[
                "projected_epoch_min"
            ],
            2,
        ),
        "min",
    )

    print(
        "peak allocated:",
        round(
            result[
                "peak_allocated_gb"
            ],
            2,
        ),
        "GB",
    )

    print(
        "attention gradients:",
        result[
            "attention_grad_params"
        ],
    )

### Show runtime table

In [ ]:
benchmark_df = pd.DataFrame(
    benchmark_results
)


display(
    benchmark_df[
        [
            "variant",
            "sec_per_step",
            "projected_epoch_min",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "attention_grad_params",
        ]
    ]
)

### Calculate slowdown on R18 None

In [ ]:
baseline_step_time = float(
    benchmark_df.loc[
        benchmark_df[
            "variant"
        ]
        == "r18_none",

        "sec_per_step",
    ].iloc[0]
)


benchmark_df[
    "runtime_multiplier"
] = (
    benchmark_df[
        "sec_per_step"
    ]
    / baseline_step_time
)


display(
    benchmark_df[
        [
            "variant",
            "sec_per_step",
            "runtime_multiplier",
            "peak_allocated_gb",
        ]
    ]
)

### Estimate FeTriL runtime based on provided benchmark

In [ ]:
BASELINE_T5_MIN_HOURS = 5.0
BASELINE_T5_MAX_HOURS = 5.9

BASELINE_T10_MIN_HOURS = 4.3
BASELINE_T10_MAX_HOURS = 5.1

In [ ]:
benchmark_df[
    "T5_est_min_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T5_MIN_HOURS
)


benchmark_df[
    "T5_est_max_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T5_MAX_HOURS
)


benchmark_df[
    "T10_est_min_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T10_MIN_HOURS
)


benchmark_df[
    "T10_est_max_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T10_MAX_HOURS
)


display(
    benchmark_df[
        [
            "variant",
            "runtime_multiplier",
            "T5_est_min_h",
            "T5_est_max_h",
            "T10_est_min_h",
            "T10_est_max_h",
            "peak_allocated_gb",
        ]
    ]
)

In [ ]:
total_min_hours = (
    benchmark_df[
        "T5_est_min_h"
    ].sum()
    +
    benchmark_df[
        "T10_est_min_h"
    ].sum()
)


total_max_hours = (
    benchmark_df[
        "T5_est_max_h"
    ].sum()
    +
    benchmark_df[
        "T10_est_max_h"
    ].sum()
)


print(
    "Estimated FeTrIL AID GPU-hours:"
)

print(
    round(
        total_min_hours,
        1,
    ),
    "-",
    round(
        total_max_hours,
        1,
    ),
    "hours",
)

## PASS

### Test dynamic rotation 256

In [14]:
from models.pa2s import PASS


pass_args = {
    **smoke_args,
    **get_backbone_variant(
        "r18_none"
    ),

    "model_name": "pass",

    "lambda_fkd": 10,
    "lambda_proto": 10,
    "temp": 0.1,

    "epochs": 1,

    "lr": 0.001,
    "weight_decay": 2e-4,

    "step_size": 45,
    "gamma": 0.1,

    "save_checkpoints": False,
}


pass_learner = PASS(
    pass_args
)


x = torch.randn(
    2,
    3,
    256,
    256,
)

y = torch.tensor(
    [3, 7],
    dtype=torch.long,
)


rx, ry = (
    pass_learner
    ._expand_rotations(
        x,
        y,
    )
)


print(
    "Input:",
    x.shape
)

print(
    "Rotated:",
    rx.shape
)

print(
    "Targets:",
    ry.tolist()
)


assert (
    rx.shape
    == (8, 3, 256, 256)
)

assert (
    ry.dtype
    == torch.long
)


print(
    "PASS 256 rotation: PASSED"
)

Input: torch.Size([2, 3, 256, 256])
Rotated: torch.Size([8, 3, 256, 256])
Targets: [12, 13, 14, 15, 28, 29, 30, 31]
PASS 256 rotation: PASSED


### Test AID 600x600

In [15]:
x = torch.randn(
    1,
    3,
    600,
    600,
)

y = torch.tensor(
    [5],
    dtype=torch.long,
)


rx, ry = (
    pass_learner
    ._expand_rotations(
        x,
        y,
    )
)


print(
    rx.shape
)

print(
    ry.tolist()
)


assert (
    rx.shape
    == (4, 3, 600, 600)
)

assert (
    ry.tolist()
    == [
        20,
        21,
        22,
        23,
    ]
)


print(
    "PASS 600 rotation: PASSED"
)

torch.Size([4, 3, 600, 600])
[20, 21, 22, 23]
PASS 600 rotation: PASSED


### Test classifier dimensionality

In [16]:
pass_learner = PASS(
    pass_args
)

pass_learner._network.update_fc(
    14 * 4
)


print(
    "PASS FC outputs:",
    pass_learner
    ._network
    .fc
    .out_features
)


assert (
    pass_learner
    ._network
    .fc
    .out_features
    == 56
)


print(
    "PASS rotation classifier: "
    "PASSED"
)

PASS FC outputs: 56
PASS rotation classifier: PASSED


### Test task 0 loss/backward

In [17]:
device = (
    pass_args["device"][0]
)


pass_learner._network = (
    pass_learner
    ._network
    .to(device)
)

pass_learner._network_module_ptr = (
    pass_learner._network
)

pass_learner._cur_task = 0
pass_learner._known_classes = 0
pass_learner._total_classes = 14


x = torch.randn(
    1,
    3,
    256,
    256,
    device=device,
)

y = torch.tensor(
    [2],
    dtype=torch.long,
    device=device,
)


x, y = (
    pass_learner
    ._expand_rotations(
        x,
        y,
    )
)


logits, clf, fkd, proto = (
    pass_learner
    ._compute_pass_loss(
        x,
        y,
    )
)


loss = (
    clf
    + fkd
    + proto
)


pass_learner._network.zero_grad()

loss.backward()


print(
    "Logits:",
    logits.shape
)

print(
    "Loss clf:",
    float(clf)
)

print(
    "Loss fkd:",
    float(fkd)
)

print(
    "Loss proto:",
    float(proto)
)


assert (
    logits.shape
    == (4, 56)
)

assert torch.isfinite(
    loss
)

assert (
    fkd.device
    == clf.device
)

assert (
    proto.device
    == clf.device
)


print(
    "PASS Task-0 backward: PASSED"
)

Logits: torch.Size([4, 56])
Loss clf: 8.726696014404297
Loss fkd: 0.0
Loss proto: 0.0
PASS Task-0 backward: PASSED


### Test loader policy

In [18]:
pass_train_dataset = (
    data_manager.get_dataset(
        np.arange(
            0,
            14,
        ),
        source="train",
        mode="train",
    )
)


loader = build_train_loader(
    dataset=(
        pass_train_dataset
    ),

    dataset_name="aid",

    batch_size=1,

    num_workers=0,

    seed=1993,
)


print(
    "PASS AID sampler:",
    type(
        loader.sampler
    ).__name__,
)


assert isinstance(
    loader.sampler,
    WeightedRandomSampler,
)


print(
    "PASS AID WRS: PASSED"
)

PASS AID sampler: WeightedRandomSampler
PASS AID WRS: PASSED


### Create new DataManager

In [19]:
from utils.data_manager import DataManager
from models.pa2s import PASS
from thesis.backbones.variants import (
    get_backbone_variant,
)
from trainer import _set_random

import gc
import time
import numpy as np
import torch

In [20]:
PASS_SMOKE_SEED = 1993


pass_smoke_args = {
    # =====================
    # Dataset
    # =====================
    "dataset": "aid",

    "dataset_root": str(
        PROJECT_ROOT
        / "datasets/raw/AID"
    ),

    "split_root": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993"
    ),

    "task_config_path": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993/T5_tasks.json"
    ),

    "shuffle": False,

    # =====================
    # PASS
    # =====================
    "model_name": "pass",

    "memory_size": 0,
    "fixed_memory": False,

    "lambda_fkd": 10,
    "lambda_proto": 10,

    "temp": 0.1,

    # Smoke only
    "epochs": 1,

    "lr": 0.001,

    "weight_decay": 2e-4,

    "step_size": 45,

    "gamma": 0.1,

    # Raw batch.
    # Effective rotation batch = 4.
    "batch_size": 1,

    "num_workers": 0,

    "save_checkpoints": False,

    # =====================
    # Reproducibility
    # =====================
    "seed": PASS_SMOKE_SEED,

    "device": [
        torch.device(
            "cuda:0"
            if torch.cuda.is_available()
            else "cpu"
        )
    ],

    # =====================
    # Backbone
    # =====================
    **get_backbone_variant(
        "r18_none"
    ),
}

In [21]:
_set_random(
    PASS_SMOKE_SEED
)


pass_data_manager = DataManager(
    dataset_name="aid",

    shuffle=False,

    seed=PASS_SMOKE_SEED,

    init_cls=14,

    increment=4,

    dataset_root=(
        pass_smoke_args[
            "dataset_root"
        ]
    ),

    split_root=(
        pass_smoke_args[
            "split_root"
        ]
    ),

    task_config_path=(
        pass_smoke_args[
            "task_config_path"
        ]
    ),
)


print(
    "Tasks:",
    pass_data_manager.nb_tasks
)

print(
    "Task sizes:",
    [
        pass_data_manager
        .get_task_size(i)

        for i in range(
            pass_data_manager.nb_tasks
        )
    ]
)


assert (
    pass_data_manager.nb_tasks
    == 5
)

assert [
    pass_data_manager
    .get_task_size(i)

    for i in range(5)
] == [
    14,
    4,
    4,
    4,
    4,
]


print(
    "PASS DataManager: PASSED"
)

Tasks: 5
Task sizes: [14, 4, 4, 4, 4]
PASS DataManager: PASSED


### Initialize fresh PASS

In [ ]:
pass_learner = PASS(
    pass_smoke_args
)


print(
    "Current task:",
    pass_learner._cur_task
)

print(
    "Known classes:",
    pass_learner._known_classes
)

print(
    "Feature dim:",
    pass_learner.feature_dim
)


assert (
    pass_learner._cur_task
    == -1
)

assert (
    pass_learner._known_classes
    == 0
)

assert (
    pass_learner.feature_dim
    == 512
)


print(
    "PASS initialization: PASSED"
)

In [ ]:
print(
    "cur_task:",
    pass_learner._cur_task
)

print(
    "known_classes:",
    pass_learner._known_classes
)

print(
    "total_classes:",
    pass_learner._total_classes
)

print(
    "prototypes:",
    len(
        pass_learner._protos
    )
)


assert (
    pass_learner._cur_task
    == -1
)

assert (
    pass_learner._known_classes
    == 0
)

assert (
    pass_learner._total_classes
    == 0
)

assert (
    len(
        pass_learner._protos
    )
    == 0
)


print(
    "Fresh PASS state: PASSED"
)

### Task 0 Training

In [ ]:
gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

In [ ]:
print(
    "cur_task:",
    pass_learner._cur_task
)

print(
    "known_classes:",
    pass_learner._known_classes
)

print(
    "total_classes:",
    pass_learner._total_classes
)

In [ ]:
import inspect

from models.pa2s import PASS


source = inspect.getsource(
    PASS.incremental_train
)

print(source)

print(
    "\n_cur_task += 1 occurrences:",
    source.count(
        "_cur_task += 1"
    )
)

print(
    "_cur_task = next_task occurrences:",
    source.count(
        "_cur_task = next_task"
    )
)

In [ ]:
task0_start = (
    time.perf_counter()
)


pass_learner.incremental_train(
    pass_data_manager
)


task0_seconds = (
    time.perf_counter()
    - task0_start
)


print(
    "Task 0 runtime:",
    round(
        task0_seconds / 60,
        2,
    ),
    "minutes",
)

### Evaluate task 0

In [ ]:
task0_cnn, task0_nme = (
    pass_learner.eval_task()
)


print(
    "Task 0 CNN top-1:",
    task0_cnn[
        "top1"
    ]
)

print(
    "Task 0 CNN grouped:",
    task0_cnn[
        "grouped"
    ]
)


if task0_nme is not None:

    print(
        "Task 0 NME top-1:",
        task0_nme[
            "top1"
        ]
    )


assert np.isfinite(
    task0_cnn[
        "top1"
    ]
)


print(
    "PASS Task 0 evaluation: PASSED"
)

### Record peak Task 0 VRAM

In [ ]:
if torch.cuda.is_available():

    task0_peak_allocated = (
        torch.cuda
        .max_memory_allocated()
        / 1024**3
    )

    task0_peak_reserved = (
        torch.cuda
        .max_memory_reserved()
        / 1024**3
    )


    print(
        "Task 0 peak allocated:",
        round(
            task0_peak_allocated,
            2,
        ),
        "GB",
    )

    print(
        "Task 0 peak reserved:",
        round(
            task0_peak_reserved,
            2,
        ),
        "GB",
    )

### after_task task 0

In [ ]:
pass_learner.after_task()


print(
    "Known classes:",
    pass_learner._known_classes
)

print(
    "Old network exists:",
    pass_learner._old_network
    is not None
)


old_trainable = sum(
    p.numel()

    for p in (
        pass_learner
        ._old_network
        .parameters()
    )

    if p.requires_grad
)


print(
    "Old network trainable params:",
    old_trainable
)


print(
    "Old network pointer exists:",
    hasattr(
        pass_learner,
        "old_network_module_ptr",
    )
)


assert (
    pass_learner._known_classes
    == 14
)

assert (
    pass_learner._old_network
    is not None
)

assert (
    old_trainable
    == 0
)

assert hasattr(
    pass_learner,
    "old_network_module_ptr",
)


print(
    "PASS frozen teacher: PASSED"
)

### Reset peak vram for task 1

In [ ]:
gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

### Run task 1

In [ ]:
task1_start = (
    time.perf_counter()
)


pass_learner.incremental_train(
    pass_data_manager
)


task1_seconds = (
    time.perf_counter()
    - task1_start
)


print(
    "Task 1 runtime:",
    round(
        task1_seconds / 60,
        2,
    ),
    "minutes",
)

### Validate incremental pass behavior

In [ ]:
print(
    "Current task:",
    pass_learner._cur_task
)

print(
    "Known classes:",
    pass_learner._known_classes
)

print(
    "Total classes:",
    pass_learner._total_classes
)

print(
    "FC outputs:",
    pass_learner
    ._network
    .fc
    .out_features
)

print(
    "Prototypes:",
    len(
        pass_learner._protos
    )
)

print(
    "Last clf:",
    pass_learner
    ._last_loss_clf
)

print(
    "Last FKD:",
    pass_learner
    ._last_loss_fkd
)

print(
    "Last proto:",
    pass_learner
    ._last_loss_proto
)

In [ ]:
assert (
    pass_learner._cur_task
    == 1
)

assert (
    pass_learner._known_classes
    == 14
)

assert (
    pass_learner._total_classes
    == 18
)

assert (
    pass_learner
    ._network
    .fc
    .out_features
    == 72
)

assert (
    len(
        pass_learner._protos
    )
    == 18
)

assert (
    pass_learner.exemplar_size
    == 0
)

assert np.isfinite(
    pass_learner
    ._last_loss_fkd
)

assert np.isfinite(
    pass_learner
    ._last_loss_proto
)

assert (
    pass_learner
    ._last_loss_fkd
    > 0
)

assert (
    pass_learner
    ._last_loss_proto
    > 0
)


print(
    "PASS Task 1 losses active: "
    "PASSED"
)

### Current backbone must be trainable

In [ ]:
current_trainable = sum(
    p.numel()

    for p in (
        pass_learner
        ._network
        .convnet
        .parameters()
    )

    if p.requires_grad
)


old_trainable = sum(
    p.numel()

    for p in (
        pass_learner
        ._old_network
        .convnet
        .parameters()
    )

    if p.requires_grad
)


print(
    "Current backbone trainable:",
    current_trainable
)

print(
    "Old backbone trainable:",
    old_trainable
)


assert (
    current_trainable
    > 0
)

assert (
    old_trainable
    == 0
)


print(
    "PASS student/teacher state: "
    "PASSED"
)

### Task 1 VRAM

In [ ]:
if torch.cuda.is_available():

    task1_peak_allocated = (
        torch.cuda
           .max_memory_allocated()
        / 1024**3
    )

    task1_peak_reserved = (
        torch.cuda
        .max_memory_reserved()
        / 1024**3
    )


    print(
        "Task 1 peak allocated:",
        round(
            task1_peak_allocated,
            2,
        ),
        "GB",
    )

    print(
        "Task 1 peak reserved:",
        round(
            task1_peak_reserved,
            2,
        ),
        "GB",
    )

### Evaluate task 1

In [ ]:
task1_cnn, task1_nme = (
    pass_learner.eval_task()
)


print(
    "Task 1 CNN top-1:",
    task1_cnn[
        "top1"
    ]
)

print(
    "Task 1 grouped:",
    task1_cnn[
        "grouped"
    ]
)


assert np.isfinite(
    task1_cnn[
        "top1"
    ]
)


print(
    "PASS Task 1 evaluation: PASSED"
)

### After task task 1

In [ ]:
pass_learner.after_task()

print(
    "After task:",
    pass_learner._cur_task
)

print(
    "Known classes:",
    pass_learner._known_classes
)

print(
    "Total classes:",
    pass_learner._total_classes
)


assert (
    pass_learner._known_classes
    == pass_learner._total_classes
)

### Run task 2-4

In [ ]:
pass_top1_curve = [
    task0_cnn["top1"],
    task1_cnn["top1"],
]

pass_runtime_minutes = [
    task0_seconds / 60,
    task1_seconds / 60,
]

In [ ]:
for expected_task in range(
    2,
    pass_data_manager.nb_tasks,
):

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    start = time.perf_counter()


    pass_learner.incremental_train(
        pass_data_manager
    )


    elapsed = (
        time.perf_counter()
        - start
    )


    cnn_accy, nme_accy = (
        pass_learner.eval_task()
    )


    expected_total = (
        pass_data_manager
        .get_accumulate_tasksize(
            expected_task
        )
    )


    print()
    print(
        "=" * 60
    )

    print(
        "Task:",
        expected_task
    )

    print(
        "Total classes:",
        pass_learner
        ._total_classes
    )

    print(
        "FC outputs:",
        pass_learner
        ._network
        .fc
        .out_features
    )

    print(
        "Prototypes:",
        len(
            pass_learner
            ._protos
        )
    )

    print(
        "FKD:",
        pass_learner
        ._last_loss_fkd
    )

    print(
        "Proto loss:",
        pass_learner
        ._last_loss_proto
    )

    print(
        "CNN top-1:",
        cnn_accy[
            "top1"
        ]
    )

    print(
        "Runtime:",
        round(
            elapsed / 60,
            2,
        ),
        "minutes",
    )


    assert (
        pass_learner._cur_task
        == expected_task
    )

    assert (
        pass_learner._total_classes
        == expected_total
    )

    assert (
        pass_learner
        ._network
        .fc
        .out_features
        == expected_total * 4
    )

    assert (
        len(
            pass_learner
            ._protos
        )
        == expected_total
    )

    assert (
        pass_learner.exemplar_size
        == 0
    )

    assert np.isfinite(
        pass_learner
        ._last_loss_fkd
    )

    assert np.isfinite(
        pass_learner
        ._last_loss_proto
    )


    pass_top1_curve.append(
        cnn_accy[
            "top1"
        ]
    )

    pass_runtime_minutes.append(
        elapsed / 60
    )


    pass_learner.after_task()

### Final PASS smoke validation

In [ ]:
print(
    "PASS top-1 curve:",
    pass_top1_curve
)

print(
    "Runtime minutes:",
    [
        round(x, 2)

        for x in (
            pass_runtime_minutes
        )
    ]
)


smoke_aia = float(
    np.mean(
        pass_top1_curve
    )
)

smoke_final = float(
    pass_top1_curve[-1]
)


print(
    "Smoke AIA:",
    round(
        smoke_aia,
        2,
    )
)

print(
    "Smoke final:",
    round(
        smoke_final,
        2,
    )
)


assert (
    len(
        pass_top1_curve
    )
    == 5
)

assert (
    pass_learner._known_classes
    == 30
)

assert (
    pass_learner._total_classes
    == 30
)

assert (
    len(
        pass_learner._protos
    )
    == 30
)

assert (
    len(
        pass_learner._radiuses
    )
    == 30
)

assert (
    pass_learner.exemplar_size
    == 0
)


print(
    "FULL PASS AID T5 "
    "smoke test: PASSED"
)

## Benchmark on 8 backbone and AID Dataset on PASS

### Dataset benchmark

In [ ]:
pass_task0_dataset = (
    pass_data_manager.get_dataset(
        np.arange(0, 14),
        source="train",
        mode="train",
    )
)

pass_task1_dataset = (
    pass_data_manager.get_dataset(
        np.arange(14, 18),
        source="train",
        mode="train",
    )
)


print(
    "Task 0 samples:",
    len(pass_task0_dataset)
)

print(
    "Task 1 samples:",
    len(pass_task1_dataset)
)


assert (
    len(pass_task0_dataset)
    == 3656
)

print(
    "PASS benchmark datasets: PASSED"
)

### Benchmark function

In [ ]:
import gc
import time

import pandas as pd

from torch import optim
from torch.utils.data import (
    WeightedRandomSampler,
)

from thesis.backbones.variants import (
    get_all_backbone_variants,
)

from thesis.data.loaders import (
    build_train_loader,
)


PASS_BENCH_BATCH = 1

PASS_WARMUP_STEPS = 3

PASS_BENCH_STEPS = 30

In [ ]:
def benchmark_pass_variant(
    variant_name,
    backbone_config,
    phase,
    batch_size=1,
    bench_steps=30,
):
    """
    phase:
        0 = initial task
        1 = incremental task
    """

    _set_random(
        PASS_SMOKE_SEED
    )

    args = {
        **pass_smoke_args,
        **backbone_config,

        "batch_size":
            batch_size,

        "num_workers":
            0,
    }

    learner = PASS(args)

    device = args[
        "device"
    ][0]

    # =====================================
    # Task 0 computational state
    # =====================================
    if phase == 0:

        learner._cur_task = 0
        learner._known_classes = 0
        learner._total_classes = 14

        learner._network.update_fc(
            14 * 4
        )

        dataset = (
            pass_task0_dataset
        )

    # =====================================
    # Task 1 computational state
    # =====================================
    elif phase == 1:

        # Build Task-0 classifier first.
        learner._network.update_fc(
            14 * 4
        )

        learner._network = (
            learner._network.to(
                device
            )
        )

        # Frozen teacher = state after Task 0.
        learner._old_network = (
            learner._network
            .copy()
            .freeze()
        )

        learner._old_network = (
            learner._old_network.to(
                device
            )
        )

        learner._old_network.eval()

        learner.old_network_module_ptr = (
            learner._old_network
        )

        # Expand student for Task 1.
        learner._network.update_fc(
            18 * 4
        )

        learner._cur_task = 1
        learner._known_classes = 14
        learner._total_classes = 18

        # Synthetic prototype state.
        # Hanya untuk benchmark compute path.
        rng = np.random.RandomState(
            PASS_SMOKE_SEED
        )

        learner._protos = [
            rng.normal(
                size=512
            ).astype(
                np.float32
            )

            for _ in range(14)
        ]

        learner._radius = 1.0

        dataset = (
            pass_task1_dataset
        )

    else:

        raise ValueError(
            "phase must be 0 or 1"
        )


    learner._network = (
        learner._network.to(
            device
        )
    )

    learner._network_module_ptr = (
        learner._network
    )

    learner._network.train()


    loader = build_train_loader(
        dataset=dataset,

        dataset_name="aid",

        batch_size=batch_size,

        num_workers=0,

        seed=(
            PASS_SMOKE_SEED
            + phase
        ),
    )


    assert isinstance(
        loader.sampler,
        WeightedRandomSampler,
    )


    optimizer = optim.Adam(
        learner
        ._network
        .parameters(),

        lr=0.001,

        weight_decay=2e-4,
    )


    iterator = iter(loader)


    # =====================================
    # Single optimization step
    # =====================================
    def run_step():

        _, inputs, targets = next(
            iterator
        )

        inputs = inputs.to(
            device,
            non_blocking=True,
        )

        targets = targets.to(
            device,
            dtype=torch.long,
            non_blocking=True,
        )

        inputs, targets = (
            learner
            ._expand_rotations(
                inputs,
                targets,
            )
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        (
            logits,
            loss_clf,
            loss_fkd,
            loss_proto,
        ) = learner._compute_pass_loss(
            inputs,
            targets,
        )

        loss = (
            loss_clf
            + loss_fkd
            + loss_proto
        )

        loss.backward()

        optimizer.step()

        return (
            float(
                loss.detach().cpu()
            )
        )


    # =====================================
    # Warm-up
    # =====================================
    for _ in range(
        PASS_WARMUP_STEPS
    ):
        run_step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    # =====================================
    # Timed benchmark
    # =====================================
    start = time.perf_counter()

    last_loss = None


    for _ in range(
        bench_steps
    ):

        last_loss = (
            run_step()
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    sec_per_step = (
        elapsed
        / bench_steps
    )


    projected_epoch_min = (
        sec_per_step
        * len(loader)
        / 60
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    attention_grad_params = sum(
        1

        for name, param
        in learner
        ._network
        .named_parameters()

        if (
            ".attention."
            in name

            and

            param.grad
            is not None
        )
    )


    result = {
        "variant":
            variant_name,

        "phase":
            (
                "task0"
                if phase == 0
                else "task1"
            ),

        "batch_size":
            batch_size,

        "samples":
            len(dataset),

        "steps_per_epoch":
            len(loader),

        "sec_per_step":
            sec_per_step,

        "projected_epoch_min":
            projected_epoch_min,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "last_loss":
            last_loss,
    }


    assert np.isfinite(
        last_loss
    )


    if (
        backbone_config[
            "attention_type"
        ]
        == "none"
    ):

        assert (
            attention_grad_params
            == 0
        )

    else:

        assert (
            attention_grad_params
            > 0
        )


    if phase == 1:

        teacher_trainable = sum(
            p.numel()

            for p in (
                learner
                ._old_network
                .parameters()
            )

            if p.requires_grad
        )

        assert (
            teacher_trainable
            == 0
        )


    del learner
    del optimizer
    del loader
    del iterator

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return result

### Run 8 variants x 2 phases

In [ ]:
pass_benchmark_results = []


variants = (
    get_all_backbone_variants()
)


for variant_name, config in (
    variants.items()
):

    for phase in [
        0,
        1,
    ]:

        print()
        print(
            "=" * 60
        )

        print(
            variant_name,
            "Task",
            phase,
        )


        result = (
            benchmark_pass_variant(
                variant_name,
                config,
                phase=phase,
                batch_size=(
                    PASS_BENCH_BATCH
                ),
                bench_steps=(
                    PASS_BENCH_STEPS
                ),
            )
        )


        pass_benchmark_results.append(
            result
        )


        print(
            "sec/step:",
            round(
                result[
                    "sec_per_step"
                ],
                4,
            )
        )

        print(
            "projected epoch:",
            round(
                result[
                    "projected_epoch_min"
                ],
                2,
            ),
            "min",
        )

        print(
            "peak allocated:",
            round(
                result[
                    "peak_allocated_gb"
                ],
                2,
            ),
            "GB",
        )

### Result table

In [ ]:
pass_benchmark_df = (
    pd.DataFrame(
        pass_benchmark_results
    )
)


display(
    pass_benchmark_df[
        [
            "variant",
            "phase",
            "sec_per_step",
            "projected_epoch_min",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "attention_grad_params",
        ]
    ]
)

### Find final batch size

In [ ]:
capacity_results = []


heavy_config = (
    get_backbone_variant(
        "r34_procbam"
    )
)


for batch in [
    2,
    4,
]:

    print()
    print(
        "=" * 60
    )

    print(
        "Testing raw batch:",
        batch
    )


    try:

        result = (
            benchmark_pass_variant(
                "r34_procbam",
                heavy_config,
                phase=1,
                batch_size=batch,
                bench_steps=10,
            )
        )

        capacity_results.append(
            result
        )


        print(
            "PASSED"
        )

        print(
            "sec/step:",
            round(
                result[
                    "sec_per_step"
                ],
                4,
            )
        )

        print(
            "peak allocated:",
            round(
                result[
                    "peak_allocated_gb"
                ],
                2,
            ),
            "GB",
        )

        print(
            "peak reserved:",
            round(
                result[
                    "peak_reserved_gb"
                ],
                2,
            ),
            "GB",
        )


    except RuntimeError as e:

        if (
            "out of memory"
            in str(e).lower()
        ):

            print(
                "OOM at batch",
                batch,
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        else:
            raise

## SSRE

### 8 variants backbone specialized for SSRE

In [22]:
from utils.inc_net import (
    IncrementalNet,
)

from thesis.backbones.variants import (
    get_all_ssre_backbone_variants,
    get_ssre_backbone_variant
)


ssre_base_args = {
    **pass_smoke_args,

    "model_name": "ssre",

    "lambda_fkd": 1,
    "lambda_proto": 10,
    "temp": 0.1,

    "mode":
        "parallel_adapters",

    "epochs": 1,

    "lr": 0.001,

    "weight_decay": 5e-4,

    "step_size": 45,

    "gamma": 0.1,

    "save_checkpoints": False,
}


for name, config in (
    get_all_ssre_backbone_variants()
    .items()
):

    args = {
        **ssre_base_args,
        **config,
    }

    network = IncrementalNet(
        args,
        False,
    )

    network.update_fc(
        14
    )

    print(
        name,
        network.feature_dim,
    )

    assert (
        network.feature_dim
        == 512
    )


print(
    "8 SSRE backbones: PASSED"
)

r18_none 512
r18_eca 512
r18_cbam 512
r18_procbam 512
r34_none 512
r34_eca 512
r34_cbam 512
r34_procbam 512
8 SSRE backbones: PASSED


### Count adapters

In [23]:
from thesis.backbones.resnet_ssre import (
    SSREConvBlock,
)


for name, config in (
    get_all_ssre_backbone_variants()
    .items()
):

    args = {
        **ssre_base_args,
        **config,
    }

    network = IncrementalNet(
        args,
        False,
    )

    adapter_blocks = sum(
        isinstance(
            module,
            SSREConvBlock,
        )

        for module in (
            network
            .convnet
            .modules()
        )
    )

    expected = (
        16
        if name.startswith(
            "r18"
        )
        else 32
    )


    print(
        name,
        "adapters:",
        adapter_blocks,
    )

    assert (
        adapter_blocks
        == expected
    )


print(
    "SSRE adapter count: PASSED"
)

r18_none adapters: 16
r18_eca adapters: 16
r18_cbam adapters: 16
r18_procbam adapters: 16
r34_none adapters: 32
r34_eca adapters: 32
r34_cbam adapters: 32
r34_procbam adapters: 32
SSRE adapter count: PASSED


### Native resolution forward

In [24]:
for name, config in (
    get_all_ssre_backbone_variants()
    .items()
):

    args = {
        **ssre_base_args,
        **config,
    }

    network = IncrementalNet(
        args,
        False,
    )

    network.update_fc(
        14
    )

    network.eval()

    for size, final_hw in [
        (256, 8),
        (600, 19),
    ]:

        x = torch.randn(
            1,
            3,
            size,
            size,
        )

        with torch.no_grad():

            output = network(x)

        assert (
            output[
                "features"
            ].shape
            == (1, 512)
        )

        assert (
            output[
                "fmaps"
            ][-1].shape
            == (
                1,
                512,
                final_hw,
                final_hw,
            )
        )


    print(
        name,
        ": PASSED"
    )

r18_none : PASSED
r18_eca : PASSED
r18_cbam : PASSED
r18_procbam : PASSED
r34_none : PASSED
r34_eca : PASSED
r34_cbam : PASSED
r34_procbam : PASSED


### Compression equivalence

In [25]:
args = {
    **ssre_base_args,
    **get_ssre_backbone_variant(
        "r18_none"
    ),
}


network = IncrementalNet(
    args,
    False,
)

network.update_fc(
    14
)

network.eval()


x = torch.randn(
    2,
    3,
    256,
    256,
)


network.convnet.switch(
    "parallel_adapters"
)


with torch.no_grad():

    before = (
        network(x)[
            "features"
        ].clone()
    )


# Manual compression equivalent
model_dict = (
    network.state_dict()
)


for key in list(
    model_dict.keys()
):

    if (
        ".adapter.weight"
        in key
    ):

        conv_key = (
            key.replace(
                ".adapter.weight",
                ".conv.weight",
            )
        )

        model_dict[
            conv_key
        ] = (
            model_dict[
                conv_key
            ]
            + F.pad(
                model_dict[key],
                [1, 1, 1, 1],
            )
        )

        model_dict[key] = (
            torch.zeros_like(
                model_dict[key]
            )
        )


network.load_state_dict(
    model_dict
)

network.convnet.switch(
    "normal"
)


with torch.no_grad():

    after = (
        network(x)[
            "features"
        ].clone()
    )


max_diff = (
    before - after
).abs().max().item()


print(
    "Max compression diff:",
    max_diff
)


assert (
    max_diff < 5e-4
)


print(
    "SSRE compression equivalence: "
    "PASSED"
)

Max compression diff: 0.0001373291015625
SSRE compression equivalence: PASSED


### Freeze policy test

In [26]:
from models.ssre import SSRE


args = {
    **ssre_base_args,
    **get_ssre_backbone_variant(
        "r18_procbam"
    ),
}


learner = SSRE(
    args
)

learner._network.update_fc(
    18
)

learner._cur_task = 1

learner._network_expansion()


trainable_convnet = [
    name

    for name, parameter
    in (
        learner
        ._network
        .convnet
        .named_parameters()
    )

    if parameter.requires_grad
]


print(
    "Trainable convnet params:",
    len(
        trainable_convnet
    )
)


assert (
    len(
        trainable_convnet
    )
    > 0
)

assert all(
    "adapter" in name

    for name
    in trainable_convnet
)


attention_trainable = [
    name

    for name, parameter
    in (
        learner
        ._network
        .convnet
        .named_parameters()
    )

    if (
        "attention" in name

        and

        parameter.requires_grad
    )
]


assert (
    len(
        attention_trainable
    )
    == 0
)


print(
    "SSRE incremental freeze policy: "
    "PASSED"
)

Trainable convnet params: 16
SSRE incremental freeze policy: PASSED


### Initial task attention trainability

In [27]:
learner = SSRE(
    args
)

learner._network.update_fc(
    14
)

learner._cur_task = 0

learner._network_expansion()


attention_trainable = [
    name

    for name, parameter
    in (
        learner
        ._network
        .convnet
        .named_parameters()
    )

    if (
        "attention" in name

        and

        parameter.requires_grad
    )
]


print(
    "Trainable attention params:",
    len(
        attention_trainable
    )
)


assert (
    len(
        attention_trainable
    )
    > 0
)


print(
    "SSRE Task0 attention training: "
    "PASSED"
)

Trainable attention params: 88
SSRE Task0 attention training: PASSED


### Task 0 loss/backwards

In [28]:
device = (
    ssre_base_args[
        "device"
    ][0]
)


args = {
    **ssre_base_args,
    **get_ssre_backbone_variant(
        "r18_none"
    ),
}


learner = SSRE(
    args
)

learner._cur_task = 0
learner._known_classes = 0
learner._total_classes = 14

learner._network.update_fc(
    14
)

learner._network_module_ptr = (
    learner._network
)

learner._network = (
    learner
    ._network
    .to(device)
)


x = torch.randn(
    1,
    3,
    256,
    256,
    device=device,
)

y = torch.tensor(
    [2],
    dtype=torch.long,
    device=device,
)


(
    logits,
    loss_clf,
    loss_fkd,
    loss_proto,
) = learner._compute_ssre_loss(
    x,
    y,
)


loss = (
    loss_clf
    + loss_fkd
    + loss_proto
)


learner._network.zero_grad()

loss.backward()


print(
    "Logits:",
    logits.shape
)

print(
    "Loss clf:",
    float(loss_clf)
)

print(
    "FKD:",
    float(loss_fkd)
)

print(
    "Proto:",
    float(loss_proto)
)


assert (
    logits.shape
    == (1, 14)
)

assert (
    float(loss_fkd)
    == 0.0
)

assert (
    float(loss_proto)
    == 0.0
)

assert torch.isfinite(
    loss
)


print(
    "SSRE Task0 backward: PASSED"
)

Logits: torch.Size([1, 14])
Loss clf: 23.974672317504883
FKD: 0.0
Proto: 0.0
SSRE Task0 backward: PASSED


### Smoke Config

In [29]:
from models.ssre import SSRE

from thesis.backbones.variants import (
    get_ssre_backbone_variant,
)

from torch.utils.data import (
    WeightedRandomSampler,
)

from thesis.backbones.resnet_ssre import (
    SSREConvBlock,
)

import gc
import time
import numpy as np
import torch

In [30]:
SSRE_SMOKE_SEED = 1993


ssre_smoke_args = {
    **ssre_base_args,

    **get_ssre_backbone_variant(
        "r18_none"
    ),

    "dataset": "aid",

    "epochs": 1,

    "batch_size": 1,

    "num_workers": 0,

    "save_checkpoints": False,

    "seed": SSRE_SMOKE_SEED,
}

### Datamanager Config

In [31]:
ssre_data_manager = DataManager(
    dataset_name="aid",

    shuffle=False,

    seed=SSRE_SMOKE_SEED,

    init_cls=14,

    increment=4,

    dataset_root=(
        ssre_smoke_args[
            "dataset_root"
        ]
    ),

    split_root=(
        ssre_smoke_args[
            "split_root"
        ]
    ),

    task_config_path=(
        ssre_smoke_args[
            "task_config_path"
        ]
    ),
)

In [32]:
_set_random(
    SSRE_SMOKE_SEED
)


ssre_learner = SSRE(
    ssre_smoke_args
)


assert (
    ssre_learner._cur_task
    == -1
)

assert (
    ssre_learner._known_classes
    == 0
)

assert (
    ssre_learner._total_classes
    == 0
)

assert (
    len(
        ssre_learner._protos
    )
    == 0
)


print(
    "Fresh SSRE state: PASSED"
)

Fresh SSRE state: PASSED


### Comppression validation helper

In [33]:
def get_ssre_adapter_max_abs(
    learner,
):
    values = []

    for name, param in (
        learner
        ._network
        .convnet
        .named_parameters()
    ):

        if "adapter" in name:

            values.append(
                param.detach()
                .abs()
                .max()
                .item()
            )

    if not values:
        return 0.0

    return max(values)

In [34]:
def assert_ssre_compressed(
    learner,
):
    adapter_max = (
        get_ssre_adapter_max_abs(
            learner
        )
    )

    modes = [
        module.mode

        for module in (
            learner
            ._network
            .convnet
            .modules()
        )

        if isinstance(
            module,
            SSREConvBlock,
        )
    ]

    assert len(
        modes
    ) > 0

    assert all(
        mode == "normal"
        for mode in modes
    )

    assert (
        adapter_max
        < 1e-12
    )

    return adapter_max

### Run all task 0-4

In [35]:
expected_totals = [
    14,
    18,
    22,
    26,
    30,
]


ssre_top1_curve = []

ssre_runtime_minutes = []

ssre_peak_vram = {}

ssre_prototype_counts = []

ssre_fc_dims = []


for expected_task, expected_total in enumerate(
    expected_totals
):

    print()
    print(
        "=" * 70
    )

    print(
        "STARTING SSRE TASK",
        expected_task,
    )


    # =====================================
    # State sebelum incremental_train
    # =====================================
    expected_known = (
        0
        if expected_task == 0
        else expected_totals[
            expected_task - 1
        ]
    )


    assert (
        ssre_learner._cur_task
        == expected_task - 1
    )

    assert (
        ssre_learner._known_classes
        == expected_known
    )


    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    # =====================================
    # Train task
    # =====================================
    start = (
        time.perf_counter()
    )


    ssre_learner.incremental_train(
        ssre_data_manager
    )


    elapsed = (
        time.perf_counter()
        - start
    )


    runtime_min = (
        elapsed / 60
    )


    ssre_runtime_minutes.append(
        runtime_min
    )


    # =====================================
    # Core state checks
    # =====================================
    assert (
        ssre_learner._cur_task
        == expected_task
    )

    assert (
        ssre_learner._total_classes
        == expected_total
    )

    assert (
        ssre_learner
        ._network
        .fc
        .out_features
        == expected_total
    )

    assert (
        len(
            ssre_learner._protos
        )
        == expected_total
    )

    assert (
        ssre_learner.exemplar_size
        == 0
    )


    # =====================================
    # AID must use WRS
    # =====================================
    assert isinstance(
        ssre_learner
        .train_loader
        .sampler,

        WeightedRandomSampler,
    )


    # =====================================
    # Compression must already be done
    # =====================================
    adapter_max = (
        assert_ssre_compressed(
            ssre_learner
        )
    )


    # =====================================
    # Loss checks
    # =====================================
    assert np.isfinite(
        ssre_learner
        ._last_loss_clf
    )

    assert np.isfinite(
        ssre_learner
        ._last_loss_fkd
    )

    assert np.isfinite(
        ssre_learner
        ._last_loss_proto
    )


    if expected_task == 0:

        assert (
            ssre_learner
            ._last_loss_fkd
            == 0.0
        )

        assert (
            ssre_learner
            ._last_loss_proto
            == 0.0
        )

    else:

        assert (
            ssre_learner
            ._last_loss_fkd
            > 0.0
        )

        assert (
            ssre_learner
            ._last_loss_proto
            > 0.0
        )


    # =====================================
    # VRAM
    # =====================================
    if (
        torch.cuda.is_available()
        and expected_task in {
            0,
            1,
        }
    ):

        peak_alloc = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )


        ssre_peak_vram[
            expected_task
        ] = {
            "allocated":
                peak_alloc,

            "reserved":
                peak_reserved,
        }


    # =====================================
    # Evaluation
    # =====================================
    cnn_accy, nme_accy = (
        ssre_learner.eval_task()
    )


    top1 = (
        cnn_accy["top1"]
    )


    assert np.isfinite(
        top1
    )


    ssre_top1_curve.append(
        top1
    )

    ssre_prototype_counts.append(
        len(
            ssre_learner
            ._protos
        )
    )

    ssre_fc_dims.append(
        ssre_learner
        ._network
        .fc
        .out_features
    )


    print(
        "Task:",
        expected_task
    )

    print(
        "Known before after_task:",
        ssre_learner
        ._known_classes
    )

    print(
        "Total classes:",
        ssre_learner
        ._total_classes
    )

    print(
        "FC outputs:",
        ssre_learner
        ._network
        .fc
        .out_features
    )

    print(
        "Prototypes:",
        len(
            ssre_learner
            ._protos
        )
    )

    print(
        "Adapter max after compression:",
        adapter_max
    )

    print(
        "Last clf loss:",
        ssre_learner
        ._last_loss_clf
    )

    print(
        "Last FKD:",
        ssre_learner
        ._last_loss_fkd
    )

    print(
        "Last proto:",
        ssre_learner
        ._last_loss_proto
    )

    print(
        "CNN top-1:",
        top1
    )

    print(
        "Runtime:",
        round(
            runtime_min,
            2,
        ),
        "minutes",
    )


    # =====================================
    # Commit task
    # =====================================
    ssre_learner.after_task()


    assert (
        ssre_learner
        ._known_classes
        == expected_total
    )

    assert (
        ssre_learner
        ._old_network
        is not None
    )


    old_trainable = sum(
        p.numel()

        for p in (
            ssre_learner
            ._old_network
            .parameters()
        )

        if p.requires_grad
    )


    assert (
        old_trainable
        == 0
    )


    print(
        "Known after_task:",
        ssre_learner
        ._known_classes
    )

    print(
        "Frozen teacher params:",
        old_trainable
    )


STARTING SSRE TASK 0


Task 0, Epoch 1/1 => Loss 3.458, Loss_clf 3.458, Loss_fkd 0.000, Loss_proto 0.000, Train_accy 13.48, Test_accy 7.66: 100%|██████████| 1/1 [02:29<00:00, 149.70s/it]


Task: 0
Known before after_task: 0
Total classes: 14
FC outputs: 14
Prototypes: 14
Adapter max after compression: 0.0
Last clf loss: 1.3381240367889404
Last FKD: 0.0
Last proto: 0.0
CNN top-1: 7.66
Runtime: 3.38 minutes
Known after_task: 14
Frozen teacher params: 0

STARTING SSRE TASK 1


Task 1, Epoch 1/1 => Loss 17.994, Loss_clf 0.371, Loss_fkd 1.205, Loss_proto 16.417, Train_accy 15.19, Test_accy 9.08: 100%|██████████| 1/1 [00:47<00:00, 47.16s/it]


Task: 1
Known before after_task: 14
Total classes: 18
FC outputs: 18
Prototypes: 18
Adapter max after compression: 0.0
Last clf loss: 0.00969113688915968
Last FKD: 2.701673746109009
Last proto: 3.234558582305908
CNN top-1: 9.08
Runtime: 1.01 minutes
Known after_task: 18
Frozen teacher params: 0

STARTING SSRE TASK 2


...

Task: 2
Known before after_task: 18
Total classes: 22
FC outputs: 22
Prototypes: 22
Adapter max after compression: 0.0
Last clf loss: 0.0943986102938652
Last FKD: 0.06820926815271378
Last proto: 7.358191013336182
CNN top-1: 9.89
Runtime: 1.23 minutes
Known after_task: 22
Frozen teacher params: 0

STARTING SSRE TASK 3


Task 3, Epoch 1/1 => Loss 8.784, Loss_clf 0.272, Loss_fkd 0.410, Loss_proto 8.101, Train_accy 0.00, Test_accy 10.37: 100%|██████████| 1/1 [00:59<00:00, 59.86s/it]


Task: 3
Known before after_task: 22
Total classes: 26
FC outputs: 26
Prototypes: 26
Adapter max after compression: 0.0
Last clf loss: 0.11078155785799026
Last FKD: 2.5292398929595947
Last proto: 0.12234169244766235
CNN top-1: 10.37
Runtime: 1.26 minutes
Known after_task: 26
Frozen teacher params: 0

STARTING SSRE TASK 4


Task 4, Epoch 1/1 => Loss 7.397, Loss_clf 0.168, Loss_fkd 0.043, Loss_proto 7.186, Train_accy 1.06, Test_accy 9.95: 100%|██████████| 1/1 [01:05<00:00, 65.23s/it]


Task: 4
Known before after_task: 26
Total classes: 30
FC outputs: 30
Prototypes: 30
Adapter max after compression: 0.0
Last clf loss: 0.045766349881887436
Last FKD: 0.0633615031838417
Last proto: 1.4155974388122559
CNN top-1: 9.95
Runtime: 1.37 minutes
Known after_task: 30
Frozen teacher params: 0


### Final status

In [36]:
print()
print(
    "=" * 70
)

print(
    "SSRE top-1 curve:",
    ssre_top1_curve
)

print(
    "Runtime minutes:",
    [
        round(
            value,
            2,
        )

        for value
        in ssre_runtime_minutes
    ]
)

print(
    "Prototype counts:",
    ssre_prototype_counts
)

print(
    "FC dimensions:",
    ssre_fc_dims
)

print(
    "Peak VRAM:",
    ssre_peak_vram
)


smoke_aia = float(
    np.mean(
        ssre_top1_curve
    )
)


print(
    "Smoke AIA:",
    round(
        smoke_aia,
        2,
    )
)

print(
    "Smoke final:",
    ssre_top1_curve[-1]
)


assert (
    ssre_prototype_counts
    == [
        14,
        18,
        22,
        26,
        30,
    ]
)

assert (
    ssre_fc_dims
    == [
        14,
        18,
        22,
        26,
        30,
    ]
)

assert (
    ssre_learner._known_classes
    == 30
)

assert (
    ssre_learner._total_classes
    == 30
)

assert (
    len(
        ssre_top1_curve
    )
    == 5
)


print(
    "FULL SSRE AID T5 "
    "smoke test: PASSED"
)


SSRE top-1 curve: [7.66, 9.08, 9.89, 10.37, 9.95]
Runtime minutes: [3.38, 1.01, 1.23, 1.26, 1.37]
Prototype counts: [14, 18, 22, 26, 30]
FC dimensions: [14, 18, 22, 26, 30]
Peak VRAM: {0: {'allocated': 0.5523161888122559, 'reserved': 0.576171875}, 1: {'allocated': 0.4956068992614746, 'reserved': 0.5234375}}
Smoke AIA: 9.39
Smoke final: 9.95
FULL SSRE AID T5 smoke test: PASSED


## Benchmark 8 backbone SSRE

### Prepare benchmark dataset

In [37]:
ssre_task0_dataset = (
    ssre_data_manager.get_dataset(
        np.arange(0, 14),
        source="train",
        mode="train",
    )
)

ssre_task1_dataset = (
    ssre_data_manager.get_dataset(
        np.arange(14, 18),
        source="train",
        mode="train",
    )
)


print(
    "Task 0 samples:",
    len(ssre_task0_dataset)
)

print(
    "Task 1 samples:",
    len(ssre_task1_dataset)
)

Task 0 samples: 3656
Task 1 samples: 928


### Benchmark function

In [38]:
import gc
import time
import pandas as pd

from torch import optim
from torch.utils.data import WeightedRandomSampler

from thesis.backbones.variants import (
    get_all_ssre_backbone_variants,
    get_ssre_backbone_variant,
)

from thesis.data.loaders import (
    build_train_loader,
)

In [39]:
SSRE_BENCH_BATCH = 1
SSRE_WARMUP_STEPS = 3
SSRE_BENCH_STEPS = 30

In [40]:
def benchmark_ssre_variant(
    variant_name,
    backbone_config,
    phase,
    batch_size=1,
    bench_steps=30,
):

    _set_random(
        SSRE_SMOKE_SEED
    )

    args = {
        **ssre_smoke_args,
        **backbone_config,

        "batch_size":
            batch_size,

        "num_workers":
            0,

        "epochs":
            1,

        "save_checkpoints":
            False,
    }


    learner = SSRE(
        args
    )


    # =====================================
    # PHASE 0 — initial task
    # =====================================
    if phase == 0:

        learner._cur_task = 0
        learner._known_classes = 0
        learner._total_classes = 14

        learner._network.update_fc(
            14
        )

        learner._network_module_ptr = (
            learner._network
        )

        learner._network_expansion()

        dataset = (
            ssre_task0_dataset
        )


    # =====================================
    # PHASE 1 — incremental task
    # =====================================
    elif phase == 1:

        # Build state equivalent to
        # completion of Task 0.
        learner._network.update_fc(
            14
        )

        learner._network.to(
            learner._device
        )

        # Simulate compressed Task-0 model:
        # adapter weights are zero.
        with torch.no_grad():

            for name, param in (
                learner
                ._network
                .convnet
                .named_parameters()
            ):

                if "adapter" in name:
                    param.zero_()


        learner._network.convnet.switch(
            "normal"
        )


        # Frozen Task-0 teacher.
        learner._old_network = (
            learner._network
            .copy()
            .freeze()
        )

        learner._old_network.to(
            learner._device
        )

        learner._old_network.eval()


        if hasattr(
            learner._old_network,
            "module",
        ):

            learner.old_network_module_ptr = (
                learner
                ._old_network
                .module
            )

        else:

            learner.old_network_module_ptr = (
                learner._old_network
            )


        # Student expands to 18 classes.
        learner._network.update_fc(
            18
        )

        learner._cur_task = 1
        learner._known_classes = 14
        learner._total_classes = 18

        learner._network_module_ptr = (
            learner._network
        )


        # Incremental SSRE:
        # freeze backbone except adapters.
        learner._network_expansion()


        # Synthetic old-class prototypes
        # only for exercising computation.
        rng = np.random.RandomState(
            SSRE_SMOKE_SEED
        )

        learner._protos = [
            rng.normal(
                size=512
            ).astype(
                np.float32
            )

            for _ in range(14)
        ]


        dataset = (
            ssre_task1_dataset
        )


    else:

        raise ValueError(
            "phase must be 0 or 1"
        )


    learner._network.to(
        learner._device
    )


    # IMPORTANT:
    # use SSRE's own train() behavior.
    #
    # Task0 -> model.train()
    # Task1 -> model.eval()
    learner.train()


    loader = build_train_loader(
        dataset=dataset,

        dataset_name="aid",

        batch_size=batch_size,

        num_workers=0,

        seed=(
            SSRE_SMOKE_SEED
            + phase
        ),
    )


    assert isinstance(
        loader.sampler,
        WeightedRandomSampler,
    )


    optimizer = optim.Adam(
        filter(
            lambda p:
                p.requires_grad,

            learner
            ._network
            .parameters(),
        ),

        lr=args["lr"],

        weight_decay=(
            args[
                "weight_decay"
            ]
        ),
    )


    iterator = iter(
        loader
    )


    def run_step():

        nonlocal iterator

        try:

            _, inputs, targets = next(
                iterator
            )

        except StopIteration:

            iterator = iter(
                loader
            )

            _, inputs, targets = next(
                iterator
            )


        inputs = inputs.to(
            learner._device,
            non_blocking=True,
        )

        targets = targets.to(
            learner._device,
            dtype=torch.long,
            non_blocking=True,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        (
            logits,
            loss_clf,
            loss_fkd,
            loss_proto,
        ) = learner._compute_ssre_loss(
            inputs,
            targets,
        )


        loss = (
            loss_clf
            + loss_fkd
            + loss_proto
        )


        loss.backward()

        optimizer.step()


        return float(
            loss.detach().cpu()
        )


    # =====================================
    # Warm-up
    # =====================================
    for _ in range(
        SSRE_WARMUP_STEPS
    ):

        run_step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    # =====================================
    # Timed steps
    # =====================================
    start = (
        time.perf_counter()
    )


    last_loss = None


    for _ in range(
        bench_steps
    ):

        last_loss = (
            run_step()
        )


    if torch.cuda.is_available():

        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    sec_per_step = (
        elapsed
        / bench_steps
    )


    projected_epoch_min = (
        sec_per_step
        * len(loader)
        / 60
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    # =====================================
    # Gradient diagnostics
    # =====================================
    attention_grad_params = sum(
        1

        for name, param
        in learner
        ._network
        .convnet
        .named_parameters()

        if (
            "attention" in name
            and param.grad is not None
        )
    )


    adapter_grad_params = sum(
        1

        for name, param
        in learner
        ._network
        .convnet
        .named_parameters()

        if (
            "adapter" in name
            and param.grad is not None
        )
    )


    backbone_trainable = sum(
        p.numel()

        for p in (
            learner
            ._network
            .convnet
            .parameters()
        )

        if p.requires_grad
    )


    result = {
        "variant":
            variant_name,

        "phase":
            (
                "task0"
                if phase == 0
                else "task1"
            ),

        "batch_size":
            batch_size,

        "samples":
            len(dataset),

        "steps_per_epoch":
            len(loader),

        "sec_per_step":
            sec_per_step,

        "projected_epoch_min":
            projected_epoch_min,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "adapter_grad_params":
            adapter_grad_params,

        "backbone_trainable_params":
            backbone_trainable,

        "last_loss":
            last_loss,
    }


    # =====================================
    # Assertions
    # =====================================
    assert np.isfinite(
        last_loss
    )


    assert (
        adapter_grad_params
        > 0
    )


    if phase == 0:

        if (
            backbone_config[
                "attention_type"
            ]
            == "none"
        ):

            assert (
                attention_grad_params
                == 0
            )

        else:

            assert (
                attention_grad_params
                > 0
            )


    else:

        # Attention must be frozen
        # during incremental SSRE tasks.
        assert (
            attention_grad_params
            == 0
        )

        teacher_trainable = sum(
            p.numel()

            for p in (
                learner
                ._old_network
                .parameters()
            )

            if p.requires_grad
        )

        assert (
            teacher_trainable
            == 0
        )


    del learner
    del optimizer
    del loader
    del iterator

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return result

### Run 8 variants x 2 phases

In [41]:
ssre_benchmark_results = []


variants = (
    get_all_ssre_backbone_variants()
)


for variant_name, config in (
    variants.items()
):

    for phase in [
        0,
        1,
    ]:

        print()
        print(
            "=" * 60
        )

        print(
            variant_name,
            "Task",
            phase,
        )


        result = (
            benchmark_ssre_variant(
                variant_name,
                config,

                phase=phase,

                batch_size=(
                    SSRE_BENCH_BATCH
                ),

                bench_steps=(
                    SSRE_BENCH_STEPS
                ),
            )
        )


        ssre_benchmark_results.append(
            result
        )


        print(
            "sec/step:",
            round(
                result[
                    "sec_per_step"
                ],
                4,
            )
        )

        print(
            "projected epoch:",
            round(
                result[
                    "projected_epoch_min"
                ],
                2,
            ),
            "min",
        )

        print(
            "peak allocated:",
            round(
                result[
                    "peak_allocated_gb"
                ],
                2,
            ),
            "GB",
        )

        print(
            "adapter grads:",
            result[
                "adapter_grad_params"
            ],
        )

        print(
            "attention grads:",
            result[
                "attention_grad_params"
            ],
        )


r18_none Task 0
sec/step: 0.0348
projected epoch: 2.12 min
peak allocated: 0.68 GB
adapter grads: 16
attention grads: 0

r18_none Task 1
sec/step: 0.0314
projected epoch: 0.49 min
peak allocated: 0.59 GB
adapter grads: 16
attention grads: 0

r18_eca Task 0
sec/step: 0.0368
projected epoch: 2.24 min
peak allocated: 0.7 GB
adapter grads: 16
attention grads: 8

r18_eca Task 1
sec/step: 0.0328
projected epoch: 0.51 min
peak allocated: 0.61 GB
adapter grads: 16
attention grads: 0

r18_cbam Task 0
sec/step: 0.0516
projected epoch: 3.15 min
peak allocated: 0.72 GB
adapter grads: 16
attention grads: 56

r18_cbam Task 1
sec/step: 0.0471
projected epoch: 0.73 min
peak allocated: 0.64 GB
adapter grads: 16
attention grads: 0

r18_procbam Task 0
sec/step: 0.0547
projected epoch: 3.34 min
peak allocated: 0.73 GB
adapter grads: 16
attention grads: 88

r18_procbam Task 1
sec/step: 0.0591
projected epoch: 0.91 min
peak allocated: 0.64 GB
adapter grads: 16
attention grads: 0

r34_none Task 0
sec/step: 

### Show table

In [42]:
ssre_benchmark_df = (
    pd.DataFrame(
        ssre_benchmark_results
    )
)


display(
    ssre_benchmark_df[
        [
            "variant",
            "phase",
            "sec_per_step",
            "projected_epoch_min",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "attention_grad_params",
            "adapter_grad_params",
            "backbone_trainable_params",
        ]
    ]
)

,variant,phase,sec_per_step,projected_epoch_min,peak_allocated_gb,peak_reserved_gb,attention_grad_params,adapter_grad_params,backbone_trainable_params
0,r18_none,task0,0.034773,2.118865,0.680892,0.779297,0,16,12397120
1,r18_none,task1,0.031380,0.485339,0.592597,0.617188,0,16,1220608
2,r18_eca,task0,0.036827,2.244021,0.701618,0.808594,8,16,12397156
3,r18_eca,task1,0.032818,0.507582,0.613421,0.638672,0,16,1220608
4,r18_cbam,task0,0.051641,3.146656,0.724608,0.814453,56,16,12487000
5,r18_cbam,task1,0.047070,0.728008,0.637224,0.664062,0,16,1220608
6,r18_procbam,task0,0.054738,3.335350,0.725394,0.810547,88,16,12576080
7,r18_procbam,task1,0.059066,0.913561,0.637151,0.666016,0,16,1220608
8,r34_none,task0,0.050876,3.100070,0.898633,0.992188,0,32,23627584
9,r34_none,task1,0.045474,0.703328,0.757531,0.781250,0,32,2342912


### Find safe batch size for SSRE

In [43]:
ssre_capacity_results = []


heavy_config = (
    get_ssre_backbone_variant(
        "r34_procbam"
    )
)


for phase in [
    0,
    1,
]:

    for batch in [
        2,
        4,
    ]:

        print()
        print(
            "=" * 60
        )

        print(
            "Phase:",
            phase,
            "Batch:",
            batch,
        )


        try:

            result = (
                benchmark_ssre_variant(
                    "r34_procbam",

                    heavy_config,

                    phase=phase,

                    batch_size=batch,

                    bench_steps=10,
                )
            )


            ssre_capacity_results.append(
                result
            )


            print(
                "PASSED"
            )

            print(
                "sec/step:",
                round(
                    result[
                        "sec_per_step"
                    ],
                    4,
                )
            )

            print(
                "peak allocated:",
                round(
                    result[
                        "peak_allocated_gb"
                    ],
                    2,
                ),
                "GB",
            )

            print(
                "peak reserved:",
                round(
                    result[
                        "peak_reserved_gb"
                    ],
                    2,
                ),
                "GB",
            )


        except RuntimeError as e:

            if (
                "out of memory"
                in str(e).lower()
            ):

                print(
                    "OOM:",
                    "phase",
                    phase,
                    "batch",
                    batch,
                )

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            else:

                raise


Phase: 0 Batch: 2
PASSED
sec/step: 0.128
peak allocated: 1.26 GB
peak reserved: 1.46 GB

Phase: 0 Batch: 4
PASSED
sec/step: 0.2121
peak allocated: 1.82 GB
peak reserved: 2.21 GB

Phase: 1 Batch: 2
PASSED
sec/step: 0.1182
peak allocated: 1.12 GB
peak reserved: 1.17 GB

Phase: 1 Batch: 4
PASSED
sec/step: 0.2193
peak allocated: 1.68 GB
peak reserved: 1.76 GB


## FeCAM

### Validate network lifecycle

In [44]:
from models.fecam import FeCAM

from thesis.backbones.variants import (
    get_all_backbone_variants,
)


fecam_base_args = {
    **ssre_smoke_args,

    "model_name":
        "fecam",

    "init_epochs":
        1,

    "init_lr":
        0.1,

    "init_weight_decay":
        5e-4,

    "beta":
        0.5,

    "alpha1":
        1.0,

    "alpha2":
        1.0,

    "ncm":
        False,

    "tukey":
        True,

    "diagonal":
        False,

    "per_class":
        True,

    "full_cov":
        True,

    "shrink":
        True,

    "norm_cov":
        True,

    "batch_size":
        1,

    "num_workers":
        0,
}

In [45]:
for name, config in (
    get_all_backbone_variants()
    .items()
):

    args = {
        **fecam_base_args,
        **config,
    }

    learner = FeCAM(
        args
    )

    learner._network.update_fc(
        14,
        0,
    )

    print(
        name,
        "feature_dim:",
        learner
        ._network
        .feature_dim,
        "fc:",
        learner
        ._network
        .fc
        .out_features,
    )

    assert (
        learner
        ._network
        .feature_dim
        == 512
    )

    assert (
        learner
        ._network
        .fc
        .out_features
        == 14
    )


print(
    "FeCAM 8 backbone "
    "initial FC test: PASSED"
)

r18_none feature_dim: 512 fc: 14
r18_eca feature_dim: 512 fc: 14
r18_cbam feature_dim: 512 fc: 14
r18_procbam feature_dim: 512 fc: 14
r34_none feature_dim: 512 fc: 14
r34_eca feature_dim: 512 fc: 14
r34_cbam feature_dim: 512 fc: 14
r34_procbam feature_dim: 512 fc: 14
FeCAM 8 backbone initial FC test: PASSED


In [46]:
args = {
    **fecam_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner = FeCAM(
    args
)


learner._network.update_fc(
    14,
    0,
)


print(
    "Task0 FC type:",
    type(
        learner._network.fc
    )
)


learner._network.update_fc(
    18,
    1,
)


print(
    "Task1 FC type:",
    type(
        learner._network.fc
    )
)

print(
    "Task1 outputs:",
    learner
    ._network
    .fc
    .out_features,
)


assert (
    learner
    ._network
    .fc
    .out_features
    == 18
)


print(
    "FeCAM cosine FC expansion: "
    "PASSED"
)

Task0 FC type: <class 'convs.linears.CosineLinear'>
Task1 FC type: <class 'convs.linears.SplitCosineLinear'>
Task1 outputs: 18
FeCAM cosine FC expansion: PASSED


### Structural covariance test

In [64]:
args = {
    **fecam_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner = FeCAM(
    args
)


rng = np.random.RandomState(
    1993
)


features = rng.random(
    (
        100,
        512,
    )
).astype(
    np.float64
)


features_t = torch.from_numpy(
    features
)


transformed = (
    learner
    ._tukeys_transform(
        features_t
    )
)


cov = torch.from_numpy(
    np.cov(
        transformed.numpy().T
    )
)


shrunk1 = (
    learner.shrink_cov(
        cov
    )
)

shrunk2 = (
    learner.shrink_cov(
        shrunk1
    )
)


print(
    "Transformed:",
    transformed.shape
)

print(
    "Covariance:",
    cov.shape
)

print(
    "Finite:",
    torch.isfinite(
        shrunk2
    ).all().item()
)


assert (
    transformed.shape
    == (100, 512)
)

assert (
    cov.shape
    == (512, 512)
)

assert torch.isfinite(
    shrunk2
).all()


print(
    "FeCAM covariance math: PASSED"
)

Transformed: torch.Size([100, 512])
Covariance: torch.Size([512, 512])
Finite: True
FeCAM covariance math: PASSED


### Structural Mahalanobis test

In [65]:
learner._cur_task = 1

learner._known_classes = 14
learner._total_classes = 18


learner._protos = [
    torch.rand(
        512,
        device=learner._device,
    )

    for _ in range(
        18
    )
]


learner._norm_cov_mat = [
    torch.eye(
        512
    )

    for _ in range(
        18
    )
]


vectors = np.random.RandomState(
    1993
).rand(
    4,
    512,
).astype(
    np.float32
)


distances = (
    learner._maha_dist(
        vectors
    )
)


print(
    "Distance shape:",
    distances.shape
)

print(
    "Finite:",
    np.isfinite(
        distances
    ).all()
)


assert (
    distances.shape
    == (18, 4)
)

assert np.isfinite(
    distances
).all()


print(
    "FeCAM Mahalanobis: PASSED"
)

Distance shape: (18, 4)
Finite: True
FeCAM Mahalanobis: PASSED


### Config smoke test

In [66]:
FECAM_SMOKE_SEED = 1993


fecam_smoke_args = {
    **fecam_base_args,
    **get_backbone_variant(
        "r18_none"
    ),

    "dataset": "aid",

    "init_epochs": 1,

    "batch_size": 1,

    "num_workers": 0,

    "seed": FECAM_SMOKE_SEED,

    "memory_size": 0,
    "fixed_memory": False,

    "beta": 0.5,
    "alpha1": 1.0,
    "alpha2": 1.0,

    "ncm": False,
    "tukey": True,

    "diagonal": False,
    "per_class": True,
    "full_cov": True,
    "shrink": True,
    "norm_cov": True,
}

### Config datamanager

In [67]:
fecam_data_manager = DataManager(
    dataset_name="aid",

    shuffle=False,

    seed=FECAM_SMOKE_SEED,

    init_cls=14,

    increment=4,

    dataset_root=(
        fecam_smoke_args[
            "dataset_root"
        ]
    ),

    split_root=(
        fecam_smoke_args[
            "split_root"
        ]
    ),

    task_config_path=(
        fecam_smoke_args[
            "task_config_path"
        ]
    ),
)

In [68]:
_set_random(
    FECAM_SMOKE_SEED
)

fecam_learner = FeCAM(
    fecam_smoke_args
)


assert fecam_learner._cur_task == -1
assert fecam_learner._known_classes == 0
assert fecam_learner._total_classes == 0

assert len(
    fecam_learner._protos
) == 0

assert len(
    fecam_learner._cov_mat
) == 0

assert (
    fecam_learner.exemplar_size
    == 0
)


print(
    "Fresh FeCAM state: PASSED"
)

Fresh FeCAM state: PASSED


### Helper freeze validation

In [69]:
def count_trainable_convnet_params(
    learner,
):
    return sum(
        p.numel()

        for p in (
            learner
            ._network
            .convnet
            .parameters()
        )

        if p.requires_grad
    )

In [70]:
def count_trainable_attention_params(
    learner,
):
    return sum(
        p.numel()

        for name, p in (
            learner
            ._network
            .convnet
            .named_parameters()
        )

        if (
            "attention" in name
            and p.requires_grad
        )
    )

### Run task 0-4

In [71]:
import gc
import time
import numpy as np
import torch

from torch.utils.data import (
    WeightedRandomSampler,
    SequentialSampler,
)


expected_totals = [
    14,
    18,
    22,
    26,
    30,
]


fecam_cnn_curve = []
fecam_maha_curve = []

fecam_runtime_minutes = []

fecam_prototype_counts = []
fecam_cov_counts = []
fecam_norm_cov_counts = []

fecam_fc_dims = []

fecam_peak_vram = {}


for expected_task, expected_total in enumerate(
    expected_totals
):

    print()
    print(
        "=" * 70
    )

    print(
        "STARTING FECAM TASK",
        expected_task
    )


    expected_known = (
        0
        if expected_task == 0

        else expected_totals[
            expected_task - 1
        ]
    )


    assert (
        fecam_learner._cur_task
        == expected_task - 1
    )

    assert (
        fecam_learner._known_classes
        == expected_known
    )


    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    start = (
        time.perf_counter()
    )


    fecam_learner.incremental_train(
        fecam_data_manager
    )


    elapsed = (
        time.perf_counter()
        - start
    )

    runtime_min = (
        elapsed / 60.0
    )

    fecam_runtime_minutes.append(
        runtime_min
    )


    # =====================================
    # State checks
    # =====================================
    assert (
        fecam_learner._cur_task
        == expected_task
    )

    assert (
        fecam_learner._total_classes
        == expected_total
    )


    # =====================================
    # Prototypes
    # =====================================
    proto_count = len(
        fecam_learner._protos
    )

    assert (
        proto_count
        == expected_total
    )


    # =====================================
    # Covariances
    # =====================================
    cov_count = len(
        fecam_learner._cov_mat
    )

    assert (
        cov_count
        == expected_total
    )


    norm_cov_count = len(
        fecam_learner._norm_cov_mat
    )

    assert (
        norm_cov_count
        == expected_total
    )


    # Every covariance must be 512 x 512
    assert all(
        cov.shape
        == (512, 512)

        for cov in (
            fecam_learner
            ._norm_cov_mat
        )
    )


    assert all(
        torch.isfinite(
            cov
        ).all()

        for cov in (
            fecam_learner
            ._norm_cov_mat
        )
    )


    # =====================================
    # Exemplar-free
    # =====================================
    assert (
        fecam_learner.exemplar_size
        == 0
    )


    # =====================================
    # AID optimizer-training loader
    # =====================================
    assert isinstance(
        fecam_learner
        .train_loader
        .sampler,

        WeightedRandomSampler,
    )


    # Test loader must remain deterministic.
    assert isinstance(
        fecam_learner
        .test_loader
        .sampler,

        SequentialSampler,
    )


    # =====================================
    # Freeze policy
    # =====================================
    convnet_trainable = (
        count_trainable_convnet_params(
            fecam_learner
        )
    )


    if expected_task == 0:

        assert (
            convnet_trainable
            > 0
        )

    else:

        assert (
            convnet_trainable
            == 0
        )


    # =====================================
    # FC dimensions/types
    # =====================================
    fc_outputs = (
        fecam_learner
        ._network
        .fc
        .out_features
    )

    assert (
        fc_outputs
        == expected_total
    )


    if expected_task == 0:

        from convs.linears import (
            CosineLinear,
        )

        assert isinstance(
            fecam_learner
            ._network
            .fc,

            CosineLinear,
        )

    else:

        from convs.linears import (
            SplitCosineLinear,
        )

        assert isinstance(
            fecam_learner
            ._network
            .fc,

            SplitCosineLinear,
        )


    # =====================================
    # VRAM
    # =====================================
    if (
        torch.cuda.is_available()
        and expected_task in {
            0,
            1,
        }
    ):

        fecam_peak_vram[
            expected_task
        ] = {
            "allocated":
                (
                    torch.cuda
                    .max_memory_allocated()
                    / 1024**3
                ),

            "reserved":
                (
                    torch.cuda
                    .max_memory_reserved()
                    / 1024**3
                ),
        }


    # =====================================
    # Evaluation
    # =====================================
    cnn_accy, fecam_accy = (
        fecam_learner.eval_task()
    )


    cnn_top1 = (
        cnn_accy["top1"]
    )

    fecam_top1 = (
        fecam_accy["top1"]
    )


    assert np.isfinite(
        cnn_top1
    )

    assert np.isfinite(
        fecam_top1
    )


    fecam_cnn_curve.append(
        cnn_top1
    )

    fecam_maha_curve.append(
        fecam_top1
    )


    fecam_prototype_counts.append(
        proto_count
    )

    fecam_cov_counts.append(
        cov_count
    )

    fecam_norm_cov_counts.append(
        norm_cov_count
    )

    fecam_fc_dims.append(
        fc_outputs
    )


    print(
        "Task:",
        expected_task
    )

    print(
        "Known before after_task:",
        fecam_learner
        ._known_classes
    )

    print(
        "Total classes:",
        fecam_learner
        ._total_classes
    )

    print(
        "FC outputs:",
        fc_outputs
    )

    print(
        "Prototypes:",
        proto_count
    )

    print(
        "Covariances:",
        cov_count
    )

    print(
        "Normalized covariances:",
        norm_cov_count
    )

    print(
        "Convnet trainable params:",
        convnet_trainable
    )

    print(
        "CNN top-1:",
        cnn_top1
    )

    print(
        "FeCAM top-1:",
        fecam_top1
    )

    print(
        "Runtime:",
        round(
            runtime_min,
            2,
        ),
        "minutes"
    )


    # =====================================
    # Commit task
    # =====================================
    fecam_learner.after_task()


    assert (
        fecam_learner
        ._known_classes
        == expected_total
    )


    print(
        "Known after_task:",
        fecam_learner
        ._known_classes
    )


STARTING FECAM TASK 0


Task 0, Epoch 1/1 => Loss 2.670, Train_accy 6.70, Test_accy 7.44: 100%|██████████| 1/1 [02:00<00:00, 120.54s/it]


Task: 0
Known before after_task: 0
Total classes: 14
FC outputs: 14
Prototypes: 14
Covariances: 14
Normalized covariances: 14
Convnet trainable params: 11176512
CNN top-1: 7.44
FeCAM top-1: 7.44
Runtime: 3.55 minutes
Known after_task: 14

STARTING FECAM TASK 1
Task: 1
Known before after_task: 14
Total classes: 18
FC outputs: 18
Prototypes: 18
Covariances: 18
Normalized covariances: 18
Convnet trainable params: 0
CNN top-1: 11.95
FeCAM top-1: 24.0
Runtime: 0.41 minutes
Known after_task: 18

STARTING FECAM TASK 2
Task: 2
Known before after_task: 18
Total classes: 22
FC outputs: 22
Prototypes: 22
Covariances: 22
Normalized covariances: 22
Convnet trainable params: 0
CNN top-1: 13.09
FeCAM top-1: 18.66
Runtime: 0.48 minutes
Known after_task: 22

STARTING FECAM TASK 3
Task: 3
Known before after_task: 22
Total classes: 26
FC outputs: 26
Prototypes: 26
Covariances: 26
Normalized covariances: 26
Convnet trainable params: 0
CNN top-1: 13.11
FeCAM top-1: 19.58
Runtime: 0.46 minutes
Known after_t

### Final validation

In [72]:
print()
print(
    "=" * 70
)


print(
    "FeCAM CNN curve:",
    fecam_cnn_curve
)

print(
    "FeCAM Mahalanobis curve:",
    fecam_maha_curve
)

print(
    "Runtime minutes:",
    [
        round(
            value,
            2,
        )

        for value
        in fecam_runtime_minutes
    ]
)

print(
    "Prototype counts:",
    fecam_prototype_counts
)

print(
    "Covariance counts:",
    fecam_cov_counts
)

print(
    "Normalized covariance counts:",
    fecam_norm_cov_counts
)

print(
    "FC dimensions:",
    fecam_fc_dims
)

print(
    "Peak VRAM:",
    fecam_peak_vram
)


smoke_aia = float(
    np.mean(
        fecam_maha_curve
    )
)


print(
    "Smoke FeCAM AIA:",
    round(
        smoke_aia,
        2,
    )
)

print(
    "Smoke final FeCAM:",
    fecam_maha_curve[-1]
)


assert (
    fecam_prototype_counts
    == [
        14,
        18,
        22,
        26,
        30,
    ]
)

assert (
    fecam_cov_counts
    == [
        14,
        18,
        22,
        26,
        30,
    ]
)

assert (
    fecam_norm_cov_counts
    == [
        14,
        18,
        22,
        26,
        30,
    ]
)

assert (
    fecam_fc_dims
    == [
        14,
        18,
        22,
        26,
        30,
    ]
)

assert (
    fecam_learner._known_classes
    == 30
)

assert (
    fecam_learner._total_classes
    == 30
)

assert (
    fecam_learner.exemplar_size
    == 0
)


print(
    "FULL FECAM AID T5 "
    "SMOKE TEST: PASSED"
)


FeCAM CNN curve: [7.44, 11.95, 13.09, 13.11, 11.8]
FeCAM Mahalanobis curve: [7.44, 24.0, 18.66, 19.58, 17.05]
Runtime minutes: [3.55, 0.41, 0.48, 0.46, 0.53]
Prototype counts: [14, 18, 22, 26, 30]
Covariance counts: [14, 18, 22, 26, 30]
Normalized covariance counts: [14, 18, 22, 26, 30]
FC dimensions: [14, 18, 22, 26, 30]
Peak VRAM: {0: {'allocated': 0.641232967376709, 'reserved': 0.69921875}, 1: {'allocated': 0.4788341522216797, 'reserved': 0.568359375}}
Smoke FeCAM AIA: 17.35
Smoke final FeCAM: 17.05
FULL FECAM AID T5 SMOKE TEST: PASSED


## Benchmark 8 variants with FeCAM

### Prepare dataset

In [73]:
fecam_task0_dataset = (
    fecam_data_manager.get_dataset(
        np.arange(0, 14),
        source="train",
        mode="train",
    )
)

fecam_task1_feature_dataset = (
    fecam_data_manager.get_dataset(
        np.arange(14, 18),
        source="train",
        mode="test",
    )
)


print(
    "Task 0 samples:",
    len(fecam_task0_dataset)
)

print(
    "Task 1 feature samples:",
    len(fecam_task1_feature_dataset)
)

Task 0 samples: 3656
Task 1 feature samples: 928


### Benchmark task 0 training

In [74]:
import gc
import time
import pandas as pd

from torch import optim

from thesis.data.loaders import (
    build_train_loader,
)

from thesis.backbones.variants import (
    get_all_backbone_variants,
    get_backbone_variant,
)


FECAM_BENCH_BATCH = 1
FECAM_WARMUP_STEPS = 3
FECAM_BENCH_STEPS = 30

### Benchmark function

In [75]:
def benchmark_fecam_task0(
    variant_name,
    backbone_config,
    batch_size=1,
    bench_steps=30,
):

    _set_random(
        FECAM_SMOKE_SEED
    )


    args = {
        **fecam_smoke_args,
        **backbone_config,

        "batch_size":
            batch_size,

        "num_workers":
            0,

        "init_epochs":
            1,
    }


    learner = FeCAM(
        args
    )


    learner._cur_task = 0
    learner._known_classes = 0
    learner._total_classes = 14


    learner._network.update_fc(
        14,
        0,
    )


    learner._network.to(
        learner._device
    )

    learner._network.train()


    loader = build_train_loader(
        dataset=(
            fecam_task0_dataset
        ),

        dataset_name="aid",

        batch_size=batch_size,

        num_workers=0,

        seed=FECAM_SMOKE_SEED,
    )


    optimizer = optim.SGD(
        filter(
            lambda p:
                p.requires_grad,

            learner
            ._network
            .parameters(),
        ),

        lr=args[
            "init_lr"
        ],

        momentum=0.9,

        weight_decay=args[
            "init_weight_decay"
        ],
    )


    iterator = iter(
        loader
    )


    def run_step():

        nonlocal iterator

        try:
            _, inputs, targets = next(
                iterator
            )

        except StopIteration:
            iterator = iter(
                loader
            )

            _, inputs, targets = next(
                iterator
            )


        inputs = inputs.to(
            learner._device,
            non_blocking=True,
        )

        targets = targets.to(
            learner._device,
            dtype=torch.long,
            non_blocking=True,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = (
            learner
            ._network(
                inputs
            )["logits"]
        )


        loss = (
            F.cross_entropy(
                logits,
                targets,
            )
        )


        loss.backward()

        optimizer.step()


        return float(
            loss.detach().cpu()
        )


    # Warm-up
    for _ in range(
        FECAM_WARMUP_STEPS
    ):
        run_step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    start = (
        time.perf_counter()
    )


    last_loss = None


    for _ in range(
        bench_steps
    ):

        last_loss = (
            run_step()
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    sec_per_step = (
        elapsed
        / bench_steps
    )


    projected_epoch_min = (
        sec_per_step
        * len(loader)
        / 60.0
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    attention_grad_params = sum(
        1

        for name, param
        in learner
        ._network
        .convnet
        .named_parameters()

        if (
            "attention" in name
            and param.grad is not None
        )
    )


    assert np.isfinite(
        last_loss
    )


    if (
        backbone_config[
            "attention_type"
        ]
        == "none"
    ):

        assert (
            attention_grad_params
            == 0
        )

    else:

        assert (
            attention_grad_params
            > 0
        )


    result = {
        "variant":
            variant_name,

        "batch_size":
            batch_size,

        "sec_per_step":
            sec_per_step,

        "projected_epoch_min":
            projected_epoch_min,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "last_loss":
            last_loss,
    }


    del learner
    del optimizer
    del loader
    del iterator

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return result

### Run 8 backbone

In [76]:
fecam_benchmark_results = []


for variant_name, config in (
    get_all_backbone_variants()
    .items()
):

    print()
    print(
        "=" * 60
    )

    print(
        variant_name
    )


    result = (
        benchmark_fecam_task0(
            variant_name,
            config,

            batch_size=(
                FECAM_BENCH_BATCH
            ),

            bench_steps=(
                FECAM_BENCH_STEPS
            ),
        )
    )


    fecam_benchmark_results.append(
        result
    )


    print(
        "sec/step:",
        round(
            result[
                "sec_per_step"
            ],
            6,
        )
    )

    print(
        "projected epoch:",
        round(
            result[
                "projected_epoch_min"
            ],
            2,
        ),
        "min",
    )

    print(
        "peak allocated:",
        round(
            result[
                "peak_allocated_gb"
            ],
            3,
        ),
        "GB",
    )

    print(
        "attention grads:",
        result[
            "attention_grad_params"
        ],
    )


r18_none
sec/step: 0.030364
projected epoch: 1.85 min
peak allocated: 0.711 GB
attention grads: 0

r18_eca
sec/step: 0.034344
projected epoch: 2.09 min
peak allocated: 0.729 GB
attention grads: 8

r18_cbam
sec/step: 0.044417
projected epoch: 2.71 min
peak allocated: 0.75 GB
attention grads: 56

r18_procbam
sec/step: 0.051563
projected epoch: 3.14 min
peak allocated: 0.751 GB
attention grads: 88

r34_none
sec/step: 0.040828
projected epoch: 2.49 min
peak allocated: 0.872 GB
attention grads: 0

r34_eca
sec/step: 0.047213
projected epoch: 2.88 min
peak allocated: 0.907 GB
attention grads: 16

r34_cbam
sec/step: 0.075082
projected epoch: 4.57 min
peak allocated: 0.948 GB
attention grads: 112

r34_procbam
sec/step: 0.086937
projected epoch: 5.3 min
peak allocated: 0.951 GB
attention grads: 176


### Show table

In [77]:
fecam_benchmark_df = (
    pd.DataFrame(
        fecam_benchmark_results
    )
)


display(
    fecam_benchmark_df[
        [
            "variant",
            "sec_per_step",
            "projected_epoch_min",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "attention_grad_params",
        ]
    ]
)

,variant,sec_per_step,projected_epoch_min,peak_allocated_gb,peak_reserved_gb,attention_grad_params
0,r18_none,0.030364,1.850177,0.710549,0.791016,0
1,r18_eca,0.034344,2.092680,0.729241,0.810547,8
2,r18_cbam,0.044417,2.706484,0.749764,0.832031,56
3,r18_procbam,0.051563,3.141894,0.750695,0.833984,88
4,r34_none,0.040828,2.487781,0.872398,0.998047,0
5,r34_eca,0.047213,2.876857,0.906588,1.029297,16
6,r34_cbam,0.075082,4.574992,0.948229,1.052734,112
7,r34_procbam,0.086937,5.297343,0.951383,1.056641,176


### Capacity test

In [78]:
heavy_config = (
    get_backbone_variant(
        "r34_procbam"
    )
)


fecam_capacity_results = []


for batch in [
    2,
    4,
    8,
]:

    print()
    print(
        "=" * 60
    )

    print(
        "FeCAM R34 ProCBAM",
        "batch:",
        batch,
    )


    try:

        result = (
            benchmark_fecam_task0(
                "r34_procbam",

                heavy_config,

                batch_size=batch,

                bench_steps=10,
            )
        )


        fecam_capacity_results.append(
            result
        )


        print(
            "PASSED"
        )

        print(
            "sec/step:",
            round(
                result[
                    "sec_per_step"
                ],
                4,
            )
        )

        print(
            "peak allocated:",
            round(
                result[
                    "peak_allocated_gb"
                ],
                2,
            ),
            "GB",
        )

        print(
            "peak reserved:",
            round(
                result[
                    "peak_reserved_gb"
                ],
                2,
            ),
            "GB",
        )

        print(
            "projected epoch:",
            round(
                result[
                    "projected_epoch_min"
                ],
                2,
            ),
            "min",
        )


    except RuntimeError as e:

        if (
            "out of memory"
            in str(e).lower()
        ):

            print(
                "OOM at batch",
                batch
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        else:
            raise


FeCAM R34 ProCBAM batch: 2
PASSED
sec/step: 0.1073
peak allocated: 1.23 GB
peak reserved: 1.39 GB
projected epoch: 3.27 min

FeCAM R34 ProCBAM batch: 4
PASSED
sec/step: 0.1844
peak allocated: 1.8 GB
peak reserved: 2.26 GB
projected epoch: 2.81 min

FeCAM R34 ProCBAM batch: 8
PASSED
sec/step: 0.3444
peak allocated: 2.98 GB
peak reserved: 3.87 GB
projected epoch: 2.62 min


## EFC

### Structural config

In [79]:
efc_base_args = {
    **fecam_smoke_args,

    "model_name":
        "efc",

    "memory_size":
        0,

    "fixed_memory":
        False,

    "epochs_first_task":
        1,

    "epochs_next_task":
        1,

    "efc_lamb":
        10.0,

    "efc_damping":
        0.1,

    "efc_protoupdate":
        0.2,

    "efc_protobatchsize":
        64,

    "batch_size":
        1,

    "num_workers":
        0,
}

### Test 8 backbone

In [80]:
from models.efc import (
    EFC,
)


for name, config in (
    get_all_backbone_variants()
    .items()
):

    args = {
        **efc_base_args,
        **config,
    }


    learner = EFC(
        args
    )


    learner._network.add_head(
        14
    )


    print(
        name,
        "feature_dim:",
        learner
        ._network
        .feature_dim,

        "heads:",
        len(
            learner
            ._network
            .heads
        ),

        "head0:",
        learner
        ._network
        .heads[0]
        .out_features,
    )


    assert (
        learner
        ._network
        .feature_dim
        == 512
    )

    assert (
        len(
            learner
            ._network
            .heads
        )
        == 1
    )

    assert (
        learner
        ._network
        .heads[0]
        .out_features
        == 14
    )


print(
    "EFC 8-backbone "
    "structural test: PASSED"
)

r18_none feature_dim: 512 heads: 1 head0: 14
r18_eca feature_dim: 512 heads: 1 head0: 14
r18_cbam feature_dim: 512 heads: 1 head0: 14
r18_procbam feature_dim: 512 heads: 1 head0: 14
r34_none feature_dim: 512 heads: 1 head0: 14
r34_eca feature_dim: 512 heads: 1 head0: 14
r34_cbam feature_dim: 512 heads: 1 head0: 14
r34_procbam feature_dim: 512 heads: 1 head0: 14
EFC 8-backbone structural test: PASSED


In [81]:
args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner = EFC(
    args
)


learner._network.add_head(
    14
)

learner._network.add_head(
    4
)


x = torch.randn(
    2,
    3,
    600,
    600,
).to(
    learner._device
)


learner._network.to(
    learner._device
)


with torch.no_grad():

    outputs, features = (
        learner
        ._network(
            x
        )
    )


print(
    "Feature:",
    features.shape
)

print(
    "Head 0:",
    outputs[0].shape
)

print(
    "Head 1:",
    outputs[1].shape
)


assert (
    features.shape
    == (2, 512)
)

assert (
    outputs[0].shape
    == (2, 14)
)

assert (
    outputs[1].shape
    == (2, 4)
)


print(
    "EFC multi-head forward: "
    "PASSED"
)

Feature: torch.Size([2, 512])
Head 0: torch.Size([2, 14])
Head 1: torch.Size([2, 4])
EFC multi-head forward: PASSED


### Test self-rotation

In [92]:
import importlib
import models.efc

print(
    "Loaded from:",
    models.efc.__file__
)

importlib.reload(
    models.efc
)
EFC = models.efc.EFC

print(
    "Has Compute Rotations:",
    hasattr(
        EFC,
        "_compute_rotations",
    )
)

print(
    "Methods:",
    [
        name
        for name in dir(EFC)
        if "compute" in name.lower()
    ]
)

Loaded from: C:\Skripsi\Project\PyCIL\models\efc.py
Has Compute Rotations: True
Methods: ['_compute_accuracy', '_compute_drift', '_compute_efm', '_compute_rotations', '_compute_task_prototypes']


In [84]:
args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner = EFC(
    args
)


learner._task_sizes = [
    14
]


images = torch.randn(
    2,
    3,
    64,
    64,
)


targets = torch.tensor(
    [
        0,
        13,
    ],
    dtype=torch.long,
)


rot_images, rot_targets = (
    learner._compute_rotations(
        images,
        targets,
        task_id=0,
    )
)


print(
    "Rotated images:",
    rot_images.shape
)

print(
    "Rotated targets:",
    rot_targets
)

print(
    "Target min/max:",
    rot_targets.min().item(),
    rot_targets.max().item(),
)


assert (
    rot_images.shape
    == (
        6,
        3,
        64,
        64,
    )
)

assert (
    rot_targets.shape
    == (6,)
)

assert (
    rot_targets.min()
    >= 14
)

assert (
    rot_targets.max()
    <= 55
)


learner._build_rotation_classifier()


assert (
    learner
    ._rotation_classifier
    .out_features
    == 42
)


print(
    "EFC self-rotation: PASSED"
)

Rotated images: torch.Size([6, 3, 64, 64])
Rotated targets: tensor([14, 15, 16, 53, 54, 55])
Target min/max: 14 55
EFC self-rotation: PASSED


### Structural EFM test

In [85]:
from torch.utils.data import (
    DataLoader,
    TensorDataset,
)


args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner = EFC(
    args
)

learner._network.add_head(
    14
)

learner._network.to(
    learner._device
)


x = torch.randn(
    4,
    3,
    64,
    64,
)

y = torch.tensor(
    [
        0,
        1,
        2,
        3,
    ],
    dtype=torch.long,
)


loader = DataLoader(
    TensorDataset(
        x,
        y,
    ),
    batch_size=2,
    shuffle=False,
)


efm = learner._compute_efm(
    loader
)


print(
    "EFM shape:",
    efm.shape
)

print(
    "Finite:",
    torch.isfinite(
        efm
    ).all().item()
)

print(
    "Symmetry max diff:",
    (
        efm
        - efm.T
    )
    .abs()
    .max()
    .item()
)


min_eigenvalue = (
    torch.linalg.eigvalsh(
        efm.cpu()
    )
    .min()
    .item()
)


print(
    "Minimum eigenvalue:",
    min_eigenvalue
)


assert (
    efm.shape
    == (512, 512)
)

assert torch.isfinite(
    efm
).all()

assert torch.allclose(
    efm,
    efm.T,
    atol=1e-6,
    rtol=1e-6,
)

assert (
    min_eigenvalue
    > -1e-5
)


print(
    "EFC EFM structural test: "
    "PASSED"
)

EFM shape: torch.Size([512, 512])
Finite: True
Symmetry max diff: 0.0
Minimum eigenvalue: -5.028384819638632e-09
EFC EFM structural test: PASSED


### EFM Loss Test

In [86]:
learner.previous_efm = (
    efm.to(
        learner._device
    )
)


features_old = torch.randn(
    4,
    512,
    device=learner._device,
)


features_same = (
    features_old.clone()
)


loss_zero = (
    learner._efm_loss(
        features_same,
        features_old,
    )
)


features_shifted = (
    features_old
    + 0.1
    * torch.randn_like(
        features_old
    )
)


loss_shifted = (
    learner._efm_loss(
        features_shifted,
        features_old,
    )
)


print(
    "Same-feature loss:",
    loss_zero.item()
)

print(
    "Shifted-feature loss:",
    loss_shifted.item()
)


assert torch.isfinite(
    loss_zero
)

assert torch.isfinite(
    loss_shifted
)

assert (
    abs(
        loss_zero.item()
    )
    < 1e-8
)

assert (
    loss_shifted.item()
    > 0
)


print(
    "EFC EFM loss: PASSED"
)

Same-feature loss: 0.0
Shifted-feature loss: 0.5679863691329956
EFC EFM loss: PASSED


### Gaussian Prototype test

In [87]:
args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner = EFC(
    args
)


rng = torch.Generator()
rng.manual_seed(
    1993
)


features = torch.randn(
    32,
    512,
    generator=rng,
)


learner._register_gaussian(
    label=0,
    features=features,
)


print(
    "Prototype count:",
    len(
        learner._prototypes
    )
)

print(
    "Covariance shape:",
    learner
    ._prototype_covariances[0]
    .shape
)

print(
    "Gaussian labels:",
    learner
    ._prototype_labels
)


samples = (
    learner
    ._gaussians[0]
    .sample(
        (8,)
    )
)


print(
    "Sample shape:",
    samples.shape
)


assert (
    len(
        learner._prototypes
    )
    == 1
)

assert (
    learner
    ._prototypes[0]
    .shape
    == (512,)
)

assert (
    learner
    ._prototype_covariances[0]
    .shape
    == (
        512,
        512,
    )
)

assert (
    samples.shape
    == (
        8,
        512,
    )
)

assert torch.isfinite(
    samples
).all()


print(
    "EFC Gaussian prototype: "
    "PASSED"
)

Prototype count: 1
Covariance shape: torch.Size([512, 512])
Gaussian labels: [0]
Sample shape: torch.Size([8, 512])
EFC Gaussian prototype: PASSED


### Test prototype sampler

In [89]:
args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner = EFC(
    args
)


learner._task_sizes = [
    14,
    4,
]


torch.manual_seed(
    1993
)


for class_idx in range(
    14
):

    features = (
        torch.randn(
            32,
            512,
        )
        + class_idx * 0.01
    )

    learner._register_gaussian(
        class_idx,
        features,
    )


proto_x, proto_y = (
    learner
    ._sample_old_prototypes(
        task_id=1,
        num_samples=64,
    )
)


print(
    "Prototype sample:",
    proto_x.shape
)

print(
    "Labels:",
    proto_y.shape
)

print(
    "Label range:",
    proto_y.min().item(),
    proto_y.max().item()
)


assert (
    proto_x.shape
    == (
        64,
        512,
    )
)

assert (
    proto_y.shape
    == (64,)
)

assert (
    proto_y.min()
    >= 0
)

assert (
    proto_y.max()
    <= 13
)

assert torch.isfinite(
    proto_x
).all()


print(
    "EFC prototype sampling: "
    "PASSED"
)

Prototype sample: torch.Size([64, 512])
Labels: torch.Size([64])
Label range: 0 13
EFC prototype sampling: PASSED


### Test task 0 rotation criterion

In [90]:
args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner0 = EFC(
    args
)

learner0._task_sizes = [
    14
]

learner0._network.add_head(
    14
)

learner0._build_rotation_classifier()

learner0._network.to(
    learner0._device
)


images = torch.randn(
    2,
    3,
    64,
    64,
    device=learner0._device,
)

targets = torch.tensor(
    [
        0,
        13,
    ],
    dtype=torch.long,
    device=learner0._device,
)


rot_images, rot_targets = (
    learner0
    ._compute_rotations(
        images,
        targets,
        task_id=0,
    )
)


all_images = torch.cat(
    [
        images,
        rot_images.to(
            learner0._device
        ),
    ],
    dim=0,
)


all_targets = torch.cat(
    [
        targets,
        rot_targets.to(
            learner0._device
        ),
    ]
)


outputs, features = (
    learner0
    ._network(
        all_images
    )
)


outputs = (
    learner0
    ._append_rotation_logits(
        outputs,
        features,
    )
)


print(
    "Combined Task0 logits:",
    outputs[0].shape
)

print(
    "Targets:",
    all_targets.shape
)


(
    loss_cls,
    loss_efm,
    loss_proto,
) = (
    learner0
    ._train_criterion(
        outputs=outputs,

        targets=all_targets,

        task_id=0,

        features=features,

        old_features=None,

        proto_to_samples=0,

        current_batch_size=2,
    )
)


print(
    "CLS:",
    loss_cls.item()
)

print(
    "EFM:",
    loss_efm.item()
)

print(
    "Proto:",
    loss_proto.item()
)


assert (
    outputs[0].shape
    == (
        8,
        56,
    )
)

assert torch.isfinite(
    loss_cls
)

assert (
    loss_cls.item()
    > 0
)

assert (
    loss_efm.item()
    == 0
)

assert (
    loss_proto.item()
    == 0
)


print(
    "EFC Task0 criterion: "
    "PASSED"
)

Combined Task0 logits: torch.Size([8, 56])
Targets: torch.Size([8])
CLS: 4.205321788787842
EFM: 0.0
Proto: 0.0
EFC Task0 criterion: PASSED


### Test task 1 PR-ACE + EFM

In [91]:
args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner1 = EFC(
    args
)


learner1._task_sizes = [
    14,
    4,
]


learner1._network.add_head(
    14
)

learner1._network.add_head(
    4
)

learner1._network.to(
    learner1._device
)


# Fake previous EFM
learner1.previous_efm = (
    torch.eye(
        512,
        device=learner1._device,
    )
)


# Register old-class Gaussian prototypes.
torch.manual_seed(
    1993
)


for class_idx in range(
    14
):

    features_class = (
        torch.randn(
            32,
            512,
        )
        + class_idx * 0.01
    )

    learner1._register_gaussian(
        class_idx,
        features_class,
    )


# Two current samples +
# one replay sample from same current task.
features = torch.randn(
    3,
    512,
    device=learner1._device,
    requires_grad=True,
)


outputs = {
    task_id:
        head(
            features
        )

    for task_id, head in enumerate(
        learner1
        ._network
        .heads
    )
}


targets = torch.tensor(
    [
        14,
        17,
        15,
    ],
    dtype=torch.long,
    device=learner1._device,
)


old_features = (
    features[
        :2
    ]
    .detach()
    + 0.05
)


(
    loss_cls,
    loss_efm,
    loss_proto,
) = (
    learner1
    ._train_criterion(
        outputs=outputs,

        targets=targets,

        task_id=1,

        features=features,

        old_features=(
            old_features
        ),

        proto_to_samples=2,

        current_batch_size=2,
    )
)


total_loss = (
    loss_cls
    + loss_efm
    + loss_proto
)


print(
    "CLS:",
    loss_cls.item()
)

print(
    "EFM:",
    loss_efm.item()
)

print(
    "PR-ACE:",
    loss_proto.item()
)

print(
    "Total:",
    total_loss.item()
)


assert torch.isfinite(
    total_loss
)

assert (
    loss_cls.item()
    > 0
)

assert (
    loss_efm.item()
    > 0
)

assert (
    loss_proto.item()
    > 0
)


total_loss.backward()


assert (
    features.grad
    is not None
)

assert torch.isfinite(
    features.grad
).all()


print(
    "EFC Task1 PR-ACE + EFM: "
    "PASSED"
)

CLS: 1.3053221702575684
EFM: 12.927996635437012
PR-ACE: 2.71374249458313
Total: 16.94706153869629
EFC Task1 PR-ACE + EFM: PASSED


### Structural drift test

In [93]:
args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}


learner = EFC(
    args
)


learner.previous_efm = (
    torch.eye(
        512
    )
)


torch.manual_seed(
    1993
)


# Register 3 fake old classes.
for class_idx in range(
    3
):

    class_features = (
        torch.randn(
            32,
            512,
        )
        + class_idx * 0.1
    )

    learner._register_gaussian(
        class_idx,
        class_features,
    )


learner._running_prototypes = [
    p.clone()

    for p in (
        learner._prototypes
    )
]


old_features = torch.randn(
    20,
    512,
)


# Controlled feature drift.
shift = torch.full(
    (
        1,
        512,
    ),
    0.05,
)


new_features = (
    old_features
    + shift
)


drift = (
    learner._compute_drift(
        new_features,
        old_features,
    )
)


print(
    "Drift shape:",
    drift.shape
)

print(
    "Finite:",
    torch.isfinite(
        drift
    ).all().item()
)

print(
    "Mean drift:",
    drift.mean().item()
)


assert (
    drift.shape
    == (
        3,
        512,
    )
)

assert torch.isfinite(
    drift
).all()

assert (
    drift.abs()
    .sum()
    > 0
)


print(
    "EFC prototype drift math: "
    "PASSED"
)

Drift shape: torch.Size([3, 512])
Finite: True
Mean drift: 0.04999999701976776
EFC prototype drift math: PASSED


### Prototype drift update test

In [94]:
old_means = [
    p.clone()

    for p in (
        learner._prototypes
    )
]


learner._update_old_prototypes_with_drift(
    drift
)


for i in range(
    3
):

    expected = (
        old_means[i]
        + drift[i]
    )

    torch.testing.assert_close(
        learner._prototypes[i],
        expected,
        atol=1e-6,
        rtol=1e-6,
    )


    # Gaussian must now use
    # updated mean.
    torch.testing.assert_close(
        learner
        ._gaussians[i]
        .loc,

        expected,

        atol=1e-6,
        rtol=1e-6,
    )


assert (
    len(
        learner
        ._running_prototypes
    )
    == 3
)


print(
    "EFC prototype drift update: "
    "PASSED"
)

EFC prototype drift update: PASSED


### Verify zero drift

In [95]:
learner.previous_efm = (
    torch.eye(
        512
    )
)


same_features = torch.randn(
    16,
    512,
)


zero_drift = (
    learner._compute_drift(
        same_features,
        same_features,
    )
)


print(
    "Zero drift max:",
    zero_drift
    .abs()
    .max()
    .item()
)


assert torch.allclose(
    zero_drift,
    torch.zeros_like(
        zero_drift
    ),
    atol=1e-7,
    rtol=1e-7,
)


print(
    "EFC zero-drift test: "
    "PASSED"
)

Zero drift max: 0.0
EFC zero-drift test: PASSED


### Config smoke test

In [97]:
EFC_SMOKE_SEED = 1993

efc_smoke_args = {
    **efc_base_args,
    **get_backbone_variant(
        "r18_none"
    ),

    "model_name":
        "efc",

    "dataset":
        "aid",

    "memory_size":
        0,

    "fixed_memory":
        False,

    "epochs_first_task":
        1,

    "epochs_next_task":
        1,

    "batch_size":
        1,

    "num_workers":
        0,

    "seed":
        1993,

    "efc_lamb":
        10.0,

    "efc_damping":
        0.1,

    "efc_protoupdate":
        0.2,

    "efc_protobatchsize":
        64,
}

### Datamanager for EFC

In [98]:
efc_data_manager = DataManager(
    dataset_name="aid",

    shuffle=False,

    seed=EFC_SMOKE_SEED,

    init_cls=14,

    increment=4,

    dataset_root=(
        efc_smoke_args[
            "dataset_root"
        ]
    ),

    split_root=(
        efc_smoke_args[
            "split_root"
        ]
    ),

    task_config_path=(
        efc_smoke_args[
            "task_config_path"
        ]
    ),
)

### Full AID T5 Smoke

In [103]:
import gc
import time
import numpy as np
import torch

from torch.utils.data import (
    WeightedRandomSampler,
    SequentialSampler,
)


_set_random(
    1993
)


efc_learner = EFC(
    efc_smoke_args
)


expected_totals = [
    14,
    18,
    22,
    26,
    30,
]


efc_curve = []
efc_runtime_minutes = []

efc_proto_counts = []
efc_gaussian_counts = []
efc_head_counts = []
efc_efm_ranks = []

efc_peak_vram = {}


for task_id, expected_total in enumerate(
    expected_totals
):

    print()
    print(
        "=" * 70
    )

    print(
        "STARTING EFC TASK",
        task_id
    )


    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    start = (
        time.perf_counter()
    )


    efc_learner.incremental_train(
        efc_data_manager
    )


    runtime = (
        time.perf_counter()
        - start
    ) / 60.0


    efc_runtime_minutes.append(
        runtime
    )


    # =====================================
    # Core state
    # =====================================
    assert (
        efc_learner._cur_task
        == task_id
    )

    assert (
        efc_learner._total_classes
        == expected_total
    )

    assert (
        sum(
            efc_learner
            ._task_sizes
        )
        == expected_total
    )


    # =====================================
    # Actual training loader
    # =====================================
    assert isinstance(
        efc_learner
        .train_loader
        .sampler,

        WeightedRandomSampler,
    )


    assert isinstance(
        efc_learner
        .test_loader
        .sampler,

        SequentialSampler,
    )


    # =====================================
    # Multi-head structure
    # =====================================
    assert (
        len(
            efc_learner
            ._network
            .heads
        )
        == task_id + 1
    )


    head_outputs = [
        head.out_features

        for head in (
            efc_learner
            ._network
            .heads
        )
    ]


    expected_heads = (
        [
            14
        ]
        +
        [
            4
        ]
        * task_id
    )


    assert (
        head_outputs
        == expected_heads
    )


    # =====================================
    # EFM
    # =====================================
    assert (
        efc_learner
        .previous_efm
        .shape
        == (
            512,
            512,
        )
    )


    assert torch.isfinite(
        efc_learner
        .previous_efm
    ).all()


    # =====================================
    # Gaussian prototypes
    # =====================================
    assert (
        len(
            efc_learner
            ._prototypes
        )
        == expected_total
    )

    assert (
        len(
            efc_learner
            ._prototype_covariances
        )
        == expected_total
    )

    assert (
        len(
            efc_learner
            ._gaussians
        )
        == expected_total
    )


    # =====================================
    # Loss lifecycle
    # =====================================
    if task_id == 0:

        assert (
            efc_learner
            ._last_loss_efm
            == 0.0
        )

        assert (
            efc_learner
            ._last_loss_proto
            == 0.0
        )

    else:

        assert (
            efc_learner
            ._last_loss_efm
            > 0.0
        )

        assert (
            efc_learner
            ._last_loss_proto
            > 0.0
        )


    # =====================================
    # Old model structure
    # =====================================
    if task_id > 0:

        assert (
            len(
                efc_learner
                ._old_network
                .heads
            )
            == task_id
        )

        assert all(
            not p.requires_grad

            for p in (
                efc_learner
                ._old_network
                .parameters()
            )
        )


    # =====================================
    # Exemplar-free
    # =====================================
    assert (
        efc_learner.exemplar_size
        == 0
    )


    # =====================================
    # VRAM
    # =====================================
    if (
        torch.cuda.is_available()
        and task_id in {
            0,
            1,
        }
    ):

        efc_peak_vram[
            task_id
        ] = {
            "allocated":
                (
                    torch.cuda
                    .max_memory_allocated()
                    / 1024**3
                ),

            "reserved":
                (
                    torch.cuda
                    .max_memory_reserved()
                    / 1024**3
                ),
        }


    # =====================================
    # Evaluation
    # =====================================
    (
        cnn_accy,
        _,
    ) = (
        efc_learner.eval_task()
    )


    top1 = (
        cnn_accy[
            "top1"
        ]
    )


    assert np.isfinite(
        top1
    )


    efc_curve.append(
        top1
    )


    efc_proto_counts.append(
        len(
            efc_learner
            ._prototypes
        )
    )

    efc_gaussian_counts.append(
        len(
            efc_learner
            ._gaussians
        )
    )

    efc_head_counts.append(
        len(
            efc_learner
            ._network
            .heads
        )
    )

    efc_efm_ranks.append(
        efc_learner
        ._efm_rank
    )


    print(
        "Heads:",
        head_outputs
    )

    print(
        "Prototypes:",
        efc_proto_counts[-1]
    )

    print(
        "Gaussians:",
        efc_gaussian_counts[-1]
    )

    print(
        "EFM rank:",
        efc_efm_ranks[-1]
    )

    print(
        "Loss:",
        efc_learner
        ._last_loss
    )

    print(
        "CLS:",
        efc_learner
        ._last_loss_cls
    )

    print(
        "EFM:",
        efc_learner
        ._last_loss_efm
    )

    print(
        "PR-ACE:",
        efc_learner
        ._last_loss_proto
    )

    print(
        "Drift norm:",
        efc_learner
        ._last_drift_norm
    )

    print(
        "Top-1:",
        top1
    )

    print(
        "Runtime:",
        round(
            runtime,
            2,
        ),
        "min"
    )


    efc_learner.after_task()


    assert (
        efc_learner
        ._known_classes
        == expected_total
    )


STARTING EFC TASK 0
Heads: [14]
Prototypes: 14
Gaussians: 14
EFM rank: 13
Loss: 4.040932263111129
CLS: 4.040932263111129
EFM: 0.0
PR-ACE: 0.0
Drift norm: 0.0
Top-1: 14.11
Runtime: 5.88 min

STARTING EFC TASK 1
Heads: [14, 4]
Prototypes: 18
Gaussians: 18
EFM rank: 17
Loss: 4.1603930207162065
CLS: 1.311368660166346
EFM: 0.032820798974391364
PR-ACE: 2.816203565176191
Drift norm: 1.590786099433899
Top-1: 14.31
Runtime: 1.38 min

STARTING EFC TASK 2
Heads: [14, 4, 4]
Prototypes: 22
Gaussians: 22
EFM rank: 21
Loss: 4.328018438507771
CLS: 1.3391400240618607
EFM: 0.01554944343490896
PR-ACE: 2.973328969447777
Drift norm: 1.1539522409439087
Top-1: 12.05
Runtime: 1.68 min

STARTING EFC TASK 3
Heads: [14, 4, 4, 4]
Prototypes: 26
Gaussians: 26
EFM rank: 25
Loss: 4.34269678454314
CLS: 1.168856462037989
EFM: 0.04864162370926611
PR-ACE: 3.125198695808649
Drift norm: 2.0843312740325928
Top-1: 9.44
Runtime: 1.63 min

STARTING EFC TASK 4
Heads: [14, 4, 4, 4, 4]
Prototypes: 30
Gaussians: 30
EFM rank: 29


### Final summary

In [104]:
print()
print(
    "=" * 70
)


print(
    "EFC top1 curve:",
    efc_curve
)

print(
    "Runtime minutes:",
    [
        round(
            value,
            2,
        )
        for value in (
            efc_runtime_minutes
        )
    ]
)

print(
    "Prototype counts:",
    efc_proto_counts
)

print(
    "Gaussian counts:",
    efc_gaussian_counts
)

print(
    "Head counts:",
    efc_head_counts
)

print(
    "EFM ranks:",
    efc_efm_ranks
)

print(
    "Peak VRAM:",
    efc_peak_vram
)


print(
    "Smoke AIA:",
    round(
        float(
            np.mean(
                efc_curve
            )
        ),
        2,
    )
)

print(
    "Smoke final:",
    efc_curve[-1]
)


assert (
    efc_proto_counts
    == [
        14,
        18,
        22,
        26,
        30,
    ]
)

assert (
    efc_gaussian_counts
    == [
        14,
        18,
        22,
        26,
        30,
    ]
)

assert (
    efc_head_counts
    == [
        1,
        2,
        3,
        4,
        5,
    ]
)


print(
    "FULL EFC AID T5 "
    "SMOKE TEST: PASSED"
)


EFC top1 curve: [14.11, 14.31, 12.05, 9.44, 10.6]
Runtime minutes: [5.88, 1.38, 1.68, 1.63, 1.74]
Prototype counts: [14, 18, 22, 26, 30]
Gaussian counts: [14, 18, 22, 26, 30]
Head counts: [1, 2, 3, 4, 5]
EFM ranks: [13, 17, 21, 25, 29]
Peak VRAM: {0: {'allocated': 1.3206725120544434, 'reserved': 1.74609375}, 1: {'allocated': 1.059514045715332, 'reserved': 1.263671875}}
Smoke AIA: 12.1
Smoke final: 10.6
FULL EFC AID T5 SMOKE TEST: PASSED


## Benchmark 8 variants with 2 phases on EFC

### Prepare dataset

In [105]:
efc_bench_task0_dataset = (
    efc_data_manager.get_dataset(
        np.arange(
            0,
            14,
        ),
        source="train",
        mode="train",
    )
)


efc_bench_task1_dataset = (
    efc_data_manager.get_dataset(
        np.arange(
            14,
            18,
        ),
        source="train",
        mode="train",
    )
)


print(
    "Task0:",
    len(
        efc_bench_task0_dataset
    )
)

print(
    "Task1:",
    len(
        efc_bench_task1_dataset
    )
)

Task0: 3656
Task1: 928


### Helper benchmark

In [106]:
import gc
import time
import numpy as np
import pandas as pd
import torch

from torch.nn import functional as F

from thesis.data.loaders import (
    build_train_loader,
)

from thesis.backbones.variants import (
    get_all_backbone_variants,
    get_backbone_variant,
)

In [107]:
def _next_batch(
    iterator,
    loader,
):
    try:
        batch = next(
            iterator
        )

    except StopIteration:

        iterator = iter(
            loader
        )

        batch = next(
            iterator
        )

    return (
        iterator,
        batch,
    )

### Benchmark task 0

In [108]:
def benchmark_efc_task0(
    variant_name,
    backbone_config,
    batch_size=1,
    warmup_steps=3,
    bench_steps=30,
):
    _set_random(
        1993
    )


    args = {
        **efc_smoke_args,
        **backbone_config,

        "batch_size":
            batch_size,

        "num_workers":
            0,
    }


    learner = EFC(
        args
    )


    learner._cur_task = 0

    learner._known_classes = 0

    learner._total_classes = 14

    learner._task_sizes = [
        14
    ]


    learner._network.add_head(
        14
    )

    learner._network.to(
        learner._device
    )


    learner._build_rotation_classifier()


    (
        optimizer,
        _,
    ) = (
        learner._build_optimizer(
            0
        )
    )


    loader = build_train_loader(
        dataset=(
            efc_bench_task0_dataset
        ),

        dataset_name="aid",

        batch_size=batch_size,

        num_workers=0,

        seed=1993,
    )


    learner._network.train()

    learner._rotation_classifier.train()


    iterator = iter(
        loader
    )


    def run_step():

        nonlocal iterator


        (
            iterator,
            batch,
        ) = (
            _next_batch(
                iterator,
                loader,
            )
        )


        (
            images,
            targets,
        ) = (
            learner._unpack_batch(
                batch
            )
        )


        images = images.to(
            learner._device,
            non_blocking=True,
        )

        targets = targets.to(
            learner._device,
            dtype=torch.long,
            non_blocking=True,
        )


        current_batch_size = (
            images.shape[0]
        )


        (
            rotated_images,
            rotated_targets,
        ) = (
            learner._compute_rotations(
                images,
                targets,
                task_id=0,
            )
        )


        all_images = torch.cat(
            [
                images,
                rotated_images,
            ],
            dim=0,
        )


        all_targets = torch.cat(
            [
                targets,
                rotated_targets,
            ],
            dim=0,
        )


        (
            outputs,
            features,
        ) = (
            learner._network(
                all_images
            )
        )


        outputs = (
            learner
            ._append_rotation_logits(
                outputs,
                features,
            )
        )


        (
            loss_cls,
            loss_efm,
            loss_proto,
        ) = (
            learner._train_criterion(
                outputs=outputs,

                targets=all_targets,

                task_id=0,

                features=features,

                old_features=None,

                proto_to_samples=0,

                current_batch_size=(
                    current_batch_size
                ),
            )
        )


        loss = (
            loss_cls
            + loss_efm
            + loss_proto
        )


        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()


        return float(
            loss.detach()
        )


    # warmup
    for _ in range(
        warmup_steps
    ):
        run_step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    start = (
        time.perf_counter()
    )


    last_loss = None


    for _ in range(
        bench_steps
    ):

        last_loss = (
            run_step()
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    sec_per_step = (
        elapsed
        / bench_steps
    )


    projected_epoch_min = (
        sec_per_step
        * len(
            loader
        )
        / 60.0
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    attention_grad_params = sum(
        1

        for name, p
        in (
            learner
            ._network
            .convnet
            .named_parameters()
        )

        if (
            "attention" in name
            and p.grad is not None
        )
    )


    backbone_trainable = sum(
        p.numel()

        for p in (
            learner
            ._network
            .convnet
            .parameters()
        )

        if p.requires_grad
    )


    result = {
        "variant":
            variant_name,

        "phase":
            "task0",

        "batch_size":
            batch_size,

        "sec_per_step":
            sec_per_step,

        "projected_epoch_min":
            projected_epoch_min,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "backbone_trainable":
            backbone_trainable,

        "last_loss":
            last_loss,
    }


    del learner
    del optimizer
    del loader
    del iterator

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return result

### Benchmark task 1

In [109]:
def benchmark_efc_task1(
    variant_name,
    backbone_config,
    batch_size=1,
    warmup_steps=3,
    bench_steps=30,
):
    _set_random(
        1993
    )


    args = {
        **efc_smoke_args,
        **backbone_config,

        "batch_size":
            batch_size,

        "num_workers":
            0,
    }


    learner = EFC(
        args
    )


    # =====================================
    # Previous Task-0 state
    # =====================================
    learner._task_sizes = [
        14
    ]

    learner._cur_task = 0
    learner._known_classes = 14
    learner._total_classes = 14


    learner._network.add_head(
        14
    )

    learner._network.to(
        learner._device
    )


    # Snapshot old model BEFORE new head.
    learner._build_old_network()


    # =====================================
    # Expand to Task 1
    # =====================================
    learner._cur_task = 1

    learner._task_sizes.append(
        4
    )

    learner._total_classes = 18


    learner._network.add_head(
        4
    )

    learner._network.to(
        learner._device
    )


    # Representative valid EFM.
    learner.previous_efm = (
        torch.eye(
            512,
            device=learner._device,
        )
    )


    # =====================================
    # Representative Gaussian prototypes
    # for old classes.
    # =====================================
    torch.manual_seed(
        1993
    )


    for class_idx in range(
        14
    ):

        mean = (
            torch.randn(
                512
            )
            * 0.01
        )

        covariance = (
            torch.eye(
                512
            )
            * 0.05
        )


        learner._prototypes.append(
            mean.clone()
        )

        learner._prototype_covariances.append(
            covariance.clone()
        )

        learner._prototype_labels.append(
            class_idx
        )


        learner._set_gaussian(
            label=class_idx,
            mean=mean,
            covariance=covariance,
        )


    learner._running_prototypes = [
        p.clone()

        for p in (
            learner._prototypes
        )
    ]


    (
        optimizer,
        _,
    ) = (
        learner._build_optimizer(
            1
        )
    )


    loader = build_train_loader(
        dataset=(
            efc_bench_task1_dataset
        ),

        dataset_name="aid",

        batch_size=batch_size,

        num_workers=0,

        seed=1994,
    )


    learner._network.eval()

    learner._old_network.eval()


    learner._previous_batch_samples = None
    learner._previous_batch_targets = None


    iterator = iter(
        loader
    )


    def run_step():

        nonlocal iterator


        (
            iterator,
            batch,
        ) = (
            _next_batch(
                iterator,
                loader,
            )
        )


        (
            images,
            targets,
        ) = (
            learner._unpack_batch(
                batch
            )
        )


        images = images.to(
            learner._device,
            non_blocking=True,
        )

        targets = targets.to(
            learner._device,
            dtype=torch.long,
            non_blocking=True,
        )


        current_batch_size = (
            images.shape[0]
        )


        # Old representation.
        with torch.no_grad():

            (
                _,
                old_features,
            ) = (
                learner
                ._old_network(
                    images
                )
            )


        (
            proto_to_samples,
            replay_images,
            replay_targets,
        ) = (
            learner
            ._sample_current_batch_buffer(
                task_id=1,

                batch_size=batch_size,
            )
        )


        if (
            replay_images is not None
            and replay_images.shape[0] > 0
        ):

            replay_images = (
                replay_images.to(
                    learner._device,
                    dtype=images.dtype,
                )
            )

            replay_targets = (
                replay_targets.to(
                    learner._device,
                    dtype=torch.long,
                )
            )


            images_for_model = (
                torch.cat(
                    [
                        images,
                        replay_images,
                    ],
                    dim=0,
                )
            )


            targets_for_loss = (
                torch.cat(
                    [
                        targets,
                        replay_targets,
                    ],
                    dim=0,
                )
            )

        else:

            images_for_model = images

            targets_for_loss = targets


        (
            outputs,
            features,
        ) = (
            learner
            ._network(
                images_for_model
            )
        )


        (
            loss_cls,
            loss_efm,
            loss_proto,
        ) = (
            learner
            ._train_criterion(
                outputs=outputs,

                targets=targets_for_loss,

                task_id=1,

                features=features,

                old_features=(
                    old_features
                ),

                proto_to_samples=(
                    proto_to_samples
                ),

                current_batch_size=(
                    current_batch_size
                ),
            )
        )


        loss = (
            loss_cls
            + loss_efm
            + loss_proto
        )


        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()


        learner._update_current_batch_buffer(
            images,
            targets,
        )


        return (
            float(
                loss.detach()
            )
        )


    # Warmup also fills previous-batch buffer.
    for _ in range(
        warmup_steps
    ):
        run_step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    start = (
        time.perf_counter()
    )


    last_loss = None


    for _ in range(
        bench_steps
    ):

        last_loss = (
            run_step()
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    sec_per_step = (
        elapsed
        / bench_steps
    )


    projected_epoch_min = (
        sec_per_step
        * len(
            loader
        )
        / 60.0
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    attention_grad_params = sum(
        1

        for name, p
        in (
            learner
            ._network
            .convnet
            .named_parameters()
        )

        if (
            "attention" in name
            and p.grad is not None
        )
    )


    backbone_trainable = sum(
        p.numel()

        for p in (
            learner
            ._network
            .convnet
            .parameters()
        )

        if p.requires_grad
    )


    result = {
        "variant":
            variant_name,

        "phase":
            "task1",

        "batch_size":
            batch_size,

        "sec_per_step":
            sec_per_step,

        "projected_epoch_min":
            projected_epoch_min,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "backbone_trainable":
            backbone_trainable,

        "last_loss":
            last_loss,
    }


    del learner
    del optimizer
    del loader
    del iterator

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return result

### Run all 8 variants

In [110]:
efc_benchmark_results = []


for (
    variant_name,
    config,
) in (
    get_all_backbone_variants()
    .items()
):

    print()
    print(
        "=" * 70
    )

    print(
        variant_name
    )


    result_t0 = (
        benchmark_efc_task0(
            variant_name,
            config,

            batch_size=1,

            bench_steps=30,
        )
    )


    result_t1 = (
        benchmark_efc_task1(
            variant_name,
            config,

            batch_size=1,

            bench_steps=30,
        )
    )


    efc_benchmark_results.extend(
        [
            result_t0,
            result_t1,
        ]
    )


    print(
        "Task0:",
        round(
            result_t0[
                "sec_per_step"
            ],
            6,
        ),
        "sec/step |",
        round(
            result_t0[
                "peak_allocated_gb"
            ],
            3,
        ),
        "GB",
    )


    print(
        "Task1:",
        round(
            result_t1[
                "sec_per_step"
            ],
            6,
        ),
        "sec/step |",
        round(
            result_t1[
                "peak_allocated_gb"
            ],
            3,
        ),
        "GB",
    )


r18_none
Task0: 0.064746 sec/step | 1.409 GB
Task1: 0.038853 sec/step | 1.193 GB

r18_eca
Task0: 0.068935 sec/step | 1.486 GB
Task1: 0.048032 sec/step | 1.233 GB

r18_cbam
Task0: 0.084537 sec/step | 1.573 GB
Task1: 0.062634 sec/step | 1.277 GB

r18_procbam
Task0: 0.092581 sec/step | 1.575 GB
Task1: 0.07778 sec/step | 1.279 GB

r34_none
Task0: 0.107594 sec/step | 1.807 GB
Task1: 0.061656 sec/step | 1.522 GB

r34_eca
Task0: 0.117919 sec/step | 1.952 GB
Task1: 0.064192 sec/step | 1.599 GB

r34_cbam
Task0: 0.146798 sec/step | 2.109 GB
Task1: 0.112192 sec/step | 1.684 GB

r34_procbam
Task0: 0.156484 sec/step | 2.11 GB
Task1: 0.134173 sec/step | 1.688 GB


### Show table

In [111]:
efc_benchmark_df = pd.DataFrame(
    efc_benchmark_results
)


display(
    efc_benchmark_df[
        [
            "variant",
            "phase",
            "batch_size",
            "sec_per_step",
            "projected_epoch_min",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "attention_grad_params",
            "backbone_trainable",
        ]
    ]
)

,variant,phase,batch_size,sec_per_step,projected_epoch_min,peak_allocated_gb,peak_reserved_gb,attention_grad_params,backbone_trainable
0,r18_none,task0,1,0.064746,3.945187,1.409094,1.865234,0,11176512
1,r18_none,task1,1,0.038853,0.600932,1.193317,1.367188,0,11166912
2,r18_eca,task0,1,0.068935,4.200421,1.485765,1.912109,8,11176548
3,r18_eca,task1,1,0.048032,0.742894,1.232805,1.421875,8,11166948
4,r18_cbam,task0,1,0.084537,5.151098,1.572945,2.013672,56,11266392
5,r18_cbam,task1,1,0.062634,0.968736,1.277153,1.455078,40,11256776
6,r18_procbam,task0,1,0.092581,5.641296,1.574964,2.009766,88,11355472
7,r18_procbam,task1,1,0.077780,1.202995,1.279087,1.468750,72,11345856
8,r34_none,task0,1,0.107594,6.556073,1.807029,2.197266,0,21284672
9,r34_none,task1,1,0.061656,0.953620,1.522193,1.759766,0,21267648


### Capacity test R34 + ProCBAM

In [112]:
heavy_config = (
    get_backbone_variant(
        "r34_procbam"
    )
)


efc_capacity = []


for batch_size in [
    2,
    4,
]:

    print()
    print(
        "=" * 70
    )

    print(
        "EFC R34 ProCBAM "
        f"batch={batch_size}"
    )


    try:

        t0 = (
            benchmark_efc_task0(
                "r34_procbam",
                heavy_config,

                batch_size=(
                    batch_size
                ),

                warmup_steps=2,

                bench_steps=10,
            )
        )


        t1 = (
            benchmark_efc_task1(
                "r34_procbam",
                heavy_config,

                batch_size=(
                    batch_size
                ),

                warmup_steps=2,

                bench_steps=10,
            )
        )


        efc_capacity.extend(
            [
                t0,
                t1,
            ]
        )


        print(
            "Task0 PASSED:",
            round(
                t0[
                    "sec_per_step"
                ],
                4,
            ),
            "sec/step |",
            round(
                t0[
                    "peak_allocated_gb"
                ],
                2,
            ),
            "GB allocated |",
            round(
                t0[
                    "peak_reserved_gb"
                ],
                2,
            ),
            "GB reserved",
        )


        print(
            "Task1 PASSED:",
            round(
                t1[
                    "sec_per_step"
                ],
                4,
            ),
            "sec/step |",
            round(
                t1[
                    "peak_allocated_gb"
                ],
                2,
            ),
            "GB allocated |",
            round(
                t1[
                    "peak_reserved_gb"
                ],
                2,
            ),
            "GB reserved",
        )


    except torch.cuda.OutOfMemoryError:

        print(
            "OOM at batch",
            batch_size
        )

        gc.collect()

        torch.cuda.empty_cache()

        break


EFC R34 ProCBAM batch=2
Task0 PASSED: 0.2869 sec/step | 3.31 GB allocated | 4.14 GB reserved
Task1 PASSED: 0.1777 sec/step | 2.27 GB allocated | 2.84 GB reserved

EFC R34 ProCBAM batch=4
Task0 PASSED: 3.3315 sec/step | 5.69 GB allocated | 7.8 GB reserved
Task1 PASSED: 0.27 sec/step | 2.87 GB allocated | 3.98 GB reserved
